# 00 · 扩展知识


来源：[飞书课程「扩展知识」目录](https://heuqqdmbyk.feishu.cn/drive/folder/ElftfVmbXlx1VPdN6qycHogfn3d) · 整理日期：2026-09-25。

本篇收录目录下的三份讲义：CLAUDE.md 扩展机制、会话中的六层记忆模型、提示词工程进阶。正文、表格和代码按原讲义完整保留；标为“勘误”或“补充”的段落用于说明原文中影响理解和使用的问题。


## 01 - CLAUDE.md 扩展机制完全指南


原讲义：[01 - CLAUDE.md 扩展机制完全指南](https://heuqqdmbyk.feishu.cn/docx/EloadcJkfoHauPxPknJc8J1dnnf)。

本部分讨论 Claude Code 的项目指令文件及加载机制。已对照 [Claude Code 官方记忆文档](https://code.claude.com/docs/en/memory) 核对导入语法、规则文件和上下文加载，必要差异在相应小节旁注明。


> 本文档系统讲解了 Claude Code 中 CLAUDE.md 的三种扩展机制：
> 
> 1. @include 引用
> 2. .claude/rules/ 条件规则
> 3. 项目中模块子目录的CLAUDE.md。
> 
> 涵盖语法、读取规则、最佳实践和常见陷阱。


### 目录

- 一、CLAUDE.md 概述
- 二、为什么要拆分
- 三、机制一：@include 递归引用
- 四、机制二：条件规则（.claude/rules/）
- 五、机制三：嵌套附件（子目录 CLAUDE.md）
- 六、三种机制对比
- 七、路径匹配模式详解
- 八、最佳实践
- 九、常见问题与陷阱
- 十、参考资源

---


### CLAUDE.md 概述


#### 什么是 CLAUDE.md

<code>CLAUDE.md</code> 是 Claude Code（CLI 版本）的项目级配置文件。当你在项目根目录（或子目录）中创建该文件后，Claude Code 会在会话启动时自动读取其中的内容，将其作为项目上下文注入到 AI 助手的上下文中。

这意味着你可以在 <code>CLAUDE.md</code> 中写入项目的构建命令、目录结构、编码规范、技术栈说明等任何 Claude 需要了解的背景知识，从而让 AI 助手更准确地理解和操作你的项目。


#### 文件位置

| <strong>位置</strong> | <strong>加载时机</strong> | <strong>说明</strong> |
| --- | --- | --- |
| 项目根目录 CLAUDE.md | 会话启动时 | 全局配置，始终加载 |
| .claude/rules/&#42;.md | 根据 paths 配置 | 条件加载，按需触发 |
| 模块子目录 CLAUDE.md | 访问该目录文件时 | 按需加载，局部配置 |


#### 文件格式

<code>CLAUDE.md</code> 是标准 Markdown 文件，可以使用 Markdown 的所有语法：标题、列表、代码块、表格等。

对于 <code>.claude/rules/</code> 目录下的规则文件，支持在文件顶部使用 YAML Frontmatter 来配置元数据（如 <code>paths</code> 字段）。

---


### 为什么要拆分


#### 上下文窗口限制

Claude 的上下文窗口（context window）是有限的。当 <code>CLAUDE.md</code> 内容过多时：

- 会占用大量上下文 Token，留给实际代码操作的空间减少
- 模型可能忽略深层位置的指令（位置偏差）
- 导致"越改越乱"或"不听话"的情况


#### 官方建议

Anthropic 官方建议将根目录 <code>CLAUDE.md</code> 控制在 <strong>200 行以内</strong>，保持精简。


#### 拆分策略总览

| <strong>策略</strong> | <strong>是否省 Token</strong> | <strong>加载方式</strong> | <strong>适用场景</strong> |
| --- | --- | --- | --- |
| @include 引用 | 否 | 启动时全量加载 | 文档模块化组织 |
| .claude/rules/ 条件规则 | 是 | 按需加载 | 按文件路径精确控制 |
| 子目录 CLAUDE.md | 是 | 访问目录时加载 | 按功能模块/子系统划分 |


### 机制一：@include 递归引用


**勘误：**当前 [Claude Code 官方记忆文档](https://code.claude.com/docs/en/memory) 使用的是 `@path/to/import`，例如 `@./docs/api-design.md`。文档没有把 `@include 路径` 列为正式语法；也并非只能导入 `.md`，官方示例包含 README 和 package.json。当前文档给出的递归深度是四次跳转，与原文“五层”的计数表述不同。引用应写在实际 CLAUDE.md 正文里，行内代码和代码围栏中的 `@路径` 不会被当作导入。


#### 功能说明

<strong>递归引用：</strong>A引用B，B引用C一次类推...

<code>@include</code> 是最直接的"文件拆分"方式。

在项目根目录下 <code>CLAUDE.md</code> 中利用@include声明引入外部文件，被引用的文件正常编写，无特殊编写的要求。

Claude Code 会在启动时把这些文件<strong>展开并拼接</strong>到上下文中。


#### 语法

<strong>完整语法：</strong>

````text
@include ./docs/api-design.md
@include ./docs/testing-conventions.md
````

<strong>简写语法：</strong>

````text
@./docs/api-design.md
@./docs/testing-conventions.md
````


#### 路径规则

- 路径相对于 <code>CLAUDE.md</code> 所在目录解析
- 支持相对路径（<code>./</code>、<code>../</code>）
- 支持通配符（部分版本支持 <code>@include ./docs/&#42;.md</code>）


#### 嵌套规则

- 最多支持 <strong>5 层</strong>嵌套引用（A 引用 B，B 引用 C……）
- 超过嵌套层数会报错
- 内置<strong>循环检测</strong>：如果 A 引用 B、B 又引用 A，系统会检测到并跳过，不会无限递归


#### 加载行为

- <strong>启动时全量加载</strong>：所有被引用的文件内容会在会话启动时一次性展开
- <strong>等价于直接写在同一个文件中</strong>：被引用的文件内容会原样插入到引用位置，只是为了按照功能分开，方便管理和阅读
- <strong>不能节省 Token</strong>：因为所有内容最终都在上下文中


#### 示例

- 项目根目录下CLAUDE.md的引用方式

````text
#项目主配置

## 项目概述
这是一个基于 TypeScript 的全栈项目...

@include ./docs/architecture.md // 引用其他地方的文件
@include ./docs/api-conventions.md // 引用其他地方的文件
@include ./docs/testing-strategy.md // 引用其他地方的文件

## 构建命令
npm run build
````

- 被引用的文件正常书写即可，无特殊要求
- 最终 Claude 看到的完整上下文等价于把所有内容拼接到一起：


#### 适用场景

- 将大型配置文件拆分为多个逻辑文档
- 团队协作时按主题分工维护不同部分的配置
- 需要跨项目复用通用配置片段


#### 注意事项

- 被引用的文件必须是 Markdown 格式（<code>.md</code>）
- 路径错误会导致加载失败（不会报错但会静默跳过）
- 循环引用会被检测并跳过，但建议在编写时避免循环

---


### 机制二：条件规则（.claude/rules/）


#### 功能说明

项目根目录下的CLAUDE.md无需引用。只要将规则文件放入 <code>.claude/rules/</code> 目录，通过YAML Frontmatter中的 <code>paths</code> 字段限定生效范围。

只有当 Claude 正在处理匹配的文件时，这条规则才被加载进上下文。

这是官方推荐的"瘦身"最佳实践。

> 注1：YAML Frontmatter就是写在md文件最上方的内容，可以是md文件的概述，或者是md文件的使用条件。
> 
> 注2：在本文当中，YAML Frontmatter中的path，就是md文件的使用条件


#### 目录结构

````text
project项目根目录/
├── .claude/
│   └── rules/
│       ├── common-rules.md      ← 无 paths，启动时全量加载
│       ├── api-rules.md         ← 有 paths，满足要求才会加载，节约token
│       └── frontend/
│           └── component-rules.md ← 子目录分类
````


#### 语法详解


##### 有 paths 字段（按需加载）

````text
---
paths:
"src/api/**/*.ts" ：
"src/controllers/**/*.ts"  
"!**/*.test.ts"
--- 

## API 层规范
所有接口必须返回统一的 Result 格式
不允许直接抛出异常，统一通过 error 字段返回
````

> 注：解释path的作用
> 
> - 第一句：匹配 <code>src/api/</code> 下所有层级的 TypeScript 文件
> - 第二句：匹配 <code>src/controllers/</code> 下所有层级的 TypeScript 文件
> - <code>!</code> — <strong>排除符</strong>，表示"从前面匹配的结果中剔除这些"
> - 第三句：即使文件路径命中了前两条规则，只要文件名是 <code>&#42;.test.ts</code>，就<strong>不加载</strong>该规则
> 
> 整体逻辑：
> 
> - 当对话涉及的文件满足以下条件时，该规则才会被注入上下文：
> - 在 src/api/下或在 src/controllers/ 下）且（不是 &#42;.test.ts 测试文件）
> - 这是一个典型的"<strong>对业务代码生效，但对测试代码不生效</strong>"的条件配置


**勘误：**原示例缺少列表符号、包含中文冒号，不是正确的 `paths` YAML 列表。按官方文档可写为：

```yaml
---
paths:
  - "src/api/**/*.ts"
  - "src/controllers/**/*.ts"
---
```

原文关于 `!` 排除和按先后顺序覆盖的描述，不能仅凭通用 glob 或 gitignore 经验套用；当前官方页面没有给出相应保证，应验证当前版本的实际匹配结果。无效 frontmatter 还可能被忽略，使规则按无 `paths` 方式加载。


##### 无 paths 字段（全量加载）

````text
---
# 通用编码规范（无 paths 字段，启动时全量加载）
--- 


# 通用编码规范

## 代码风格
- 变量命名使用 camelCase
- 常量命名使用 UPPER_SNAKE_CASE
````

> <strong>注意</strong>：YAML Frontmatter（<code>---</code> 包裹的 YAML 块）是可选的。
> 
> 如果没有 <code>paths</code> 字段，文件行为等同于写在 <code>CLAUDE.md</code> 中——启动时全量加载。


#### paths 路径匹配模式

<code>.claude/rules/</code> 中的 <code>paths</code> 字段使用 <strong>Git 风格的路径匹配模式</strong>（glob patterns）：

| <strong>模式</strong> | <strong>说明</strong> | <strong>示例</strong> |
| --- | --- | --- |
| &#42;.ts | 匹配当前目录下所有 .ts 文件 | src/&#42;.ts 匹配 src/main.ts |
| &#42;&#42;/&#42;.ts | 匹配所有层级下的 .ts 文件 | src/&#42;&#42;/&#42;.ts 匹配 src/api/user.ts |
| !pattern | 排除匹配的文件 | !&#42;&#42;/&#42;.test.ts 排除所有测试文件 |
| dir/file | 精确匹配 | src/api/users.ts |
| dir/&#42;/file | 仅匹配一级子目录 | src/api/&#42;/routes.ts |

<strong>常见组合示例：</strong>

````text
paths:
"src/**/*.ts"            # 匹配 src 下所有 .ts 文件
"!**/*.test.ts"          # 排除所有测试文件
"docs/**/*.md"           # 匹配 docs 下所有 markdown 文件
"scripts/**/*.js"        # 匹配 scripts 下所有 js 文件
````


#### 加载规则详解

| <strong>规则文件配置</strong> | <strong>加载时机</strong> | <strong>Token 消耗</strong> |
| --- | --- | --- |
| 无 paths 字段 | 会话启动时全量加载 | 始终消耗 |
| 有 paths 字段 | Claude 读写匹配路径的文件时触发 | 仅匹配时消耗 |
| paths 匹配的文件已被处理过 | 部分版本会缓存直到上下文压缩 | 可能常驻 |


#### 规则文件命名和分类

<code>.claude/rules/</code> 支持子目录进行分类：

````text
.claude/rules/
├── common.md                  # 通用规则
├── coding-style.md            # 代码风格
├── testing.md                 # 测试规范
├── security.md                # 安全规范
└── frontend/
    ├── components.md          # 前端组件规范
    └── state-management.md    # 状态管理规范
````


#### 适用场景

- 存放只在特定场景下需要的具体约束或约定
- 按文件类型/路径精确控制规则的生效范围
- 精准分配上下文成本，节省 Token

---


### 机制三：嵌套附件（子目录 CLAUDE.md）


#### 功能说明

在项目的子目录中放置独立的 <code>CLAUDE.md</code> 文件。Claude 不会在启动时读取它们，而是<strong>在访问该目录下的文件时自动加载</strong>。

这是最"无感"的按需加载方式。


#### 目录结构示例

````text
project/
├── CLAUDE.md                ← 启动时加载（全局约定）
├── frontend/
│   ├── CLAUDE.md            ← 访问 frontend/ 时自动加载
│   └── src/
│       └── components/
│           └── CLAUDE.md    ← 访问 components/ 时自动加载
├── backend/
│   ├── CLAUDE.md            ← 访问 backend/ 时自动加载
│   └── api/
│       └── routes/
│           └── CLAUDE.md    ← 访问 routes/ 时自动加载
````


#### 触发机制

| <strong>场景</strong> | <strong>是否加载子目录配置</strong> |
| --- | --- |
| 会话启动 | 不加载子目录 CLAUDE.md |
| Claude 执行 Read("frontend/src/App.tsx") | 加载 frontend/CLAUDE.md |
| Claude 执行 Read("backend/api/routes/user.ts") | 加载 backend/CLAUDE.md + backend/api/routes/CLAUDE.md（多层叠加） |
| 会话切换到其他目录 | 已加载的子目录规则保持到压缩或切换上下文 |


#### 核心特征

| <strong>特征</strong> | <strong>说明</strong> |
| --- | --- |
| 零语法成本 | 不需要写任何引用代码，放好文件即可 |
| 多层叠加 | 访问深层目录时，从项目根到该目录路径上的所有 CLAUDE.md 都会生效 |
| 优先级 | 子目录 CLAUDE.md &gt; 父目录 CLAUDE.md &gt; 项目根目录 CLAUDE.md（更局部的规则覆盖全局） |
| 按需加载 | 只有访问对应目录的文件时才加载，不占用初始上下文 |


#### 优先级与覆盖规则

当多个层级的 <code>CLAUDE.md</code> 同时生效时：

1. <strong>所有匹配的文件内容都会被加载</strong>（不会互相覆盖）
2. <strong>后加载的优先级更高</strong>：子目录的配置在优先级上高于父目录
3. 如果子目录和父目录有冲突的指令，子目录的指令通常会覆盖父目录的对应指令


> **勘误：**官方文档说明这些文件会依次加入上下文，并不保证子目录文件像配置系统一样自动覆盖父目录的冲突指令。尽量消除冲突，通过 `/context` 检查实际加载情况；需要强制执行的约束应使用权限、沙箱或合适的 hook 等机制。


#### 适用场景

- 大型项目或 Monorepo，各子系统有独立规范
- 前端和后端互不干扰，避免无关模块的规范污染当前上下文
- 需要为不同技术栈的子项目维护独立的 AI 助手行为

---


### 三种机制对比


#### 总览对比表

| <strong>维度</strong> | <strong>@include 递归</strong> | <strong>条件规则（rules + paths）</strong> | <strong>嵌套附件（子目录 CLAUDE.md）</strong> |
| --- | --- | --- | --- |
| <strong>语法成本</strong> | 需手动写 @include | 需写 YAML frontmatter | 零语法，放文件即可 |
| <strong>是否省 Token</strong> | 否（全量加载） | 是（按需加载） | 是（按需加载） |
| <strong>适用场景</strong> | 文档模块化组织 | 按文件类型/路径精确控制 | 按功能模块/子系统划分 |
| <strong>加载触发</strong> | 启动时 | 匹配 paths 时 | 访问该目录文件时 |
| <strong>最大层级</strong> | 5 层 | 无限制 | 无限嵌套 |
| <strong>循环检测</strong> | 有 | 不适用 | 不适用 |
| <strong>优先级控制</strong> | 引用顺序 | paths 匹配顺序 | 目录层级深度 |


#### 组合使用策略

三种机制可以组合使用，形成分层的配置体系：

````text
推荐配置层次：
┌─────────────────────────────────────────────┐
│ 第1层：根目录 CLAUDE.md（<60行，全局配置）     │
│   - 构建命令                                 │
│   - 项目架构概述                             │
│   - @include 引用子目录配置（可选）           │
├─────────────────────────────────────────────┤
│ 第2层：.claude/rules/（按文件路径条件加载）    │
│   - 通用编码规范（无 paths）                  │
│   - API 规范（paths: src/api/**）            │
│   - 测试规范（paths: **/*.test.ts）          │
├─────────────────────────────────────────────┤
│ 第3层：子目录 CLAUDE.md（按目录按需加载）      │
│   - frontend/CLAUDE.md                      │
│   - backend/CLAUDE.md                       │
│   - packages/shared/CLAUDE.md               │
└─────────────────────────────────────────────┘
````

---


### Path路径匹配模式详解


#### Glob 模式速查

| <strong>模式</strong> | <strong>匹配规则</strong> | <strong>匹配示例</strong> | <strong>不匹配示例</strong> |
| --- | --- | --- | --- |
| &#42;.ts | 当前目录下的 .ts 文件 | main.ts | src/main.ts |
| &#42;&#42;/&#42;.ts | 所有层级的 .ts 文件 | src/main.ts、a/b/c.ts | main.js |
| src/&#42;&#42; | src 目录下所有内容 | src/api.ts、src/a/b/c.ts | test/src.ts |
| !&#42;.test.ts | 排除测试文件 | main.ts ✓ | main.test.ts ✗ |
| src/api/?&#42;.ts | 单字符匹配 | src/api/a.ts | src/api/ab.ts |
| {a,b}.ts | 多选一 | a.ts、b.ts | c.ts |


> **勘误：**`src/api/?*.ts` 中 `?` 匹配一个字符、`*` 再匹配零个或多个字符，因此通常也会匹配 `src/api/ab.ts`。如果只允许一个字符的文件名，应写 `src/api/?.ts`。


#### 排除规则

使用 <code>!</code> 前缀表示排除：

````text
paths:
"src/**/*.ts"        # 包含 src 下所有 ts 文件
"!**/*.test.ts"      # 排除测试文件
"!**/*.spec.ts"      # 排除 spec 文件
"!src/mocks/**"      # 排除 mock 目录
````

> <strong>注意</strong>：排除规则必须放在包含规则之后，按照从上到下的顺序匹配。


#### 路径基准

- <code>.claude/rules/</code> 中的路径匹配相对于<strong>项目根目录</strong>
- <code>@include</code> 中的路径相对于<strong>当前文件所在目录</strong>

---


### 最佳实践


#### 根目录 CLAUDE.md 瘦身原则

1. <strong>控制在 60-200 行</strong>：只保留每次会话都需要的全局信息
2. <strong>只放高频信息</strong>：构建命令、目录结构、核心技术栈
3. <strong>用 @include 引用</strong>：将长文档拆分到外部文件
4. <strong>用子目录配置</strong>：将模块特定配置放到子目录


#### .claude/rules/ 使用建议

1. <strong>按关注点拆分</strong>：将编码规范、测试规范、安全规范等拆为独立文件
2. <strong>善用 paths</strong>：为只在特定场景需要的规则配置路径限定
3. <strong>通用规则放顶层</strong>：没有 paths 的规则文件作为全局规范
4. <strong>子目录分类</strong>：使用 <code>.claude/rules/frontend/</code> 等子目录组织规则


#### 子目录 CLAUDE.md 使用建议

1. <strong>按技术栈划分</strong>：前端、后端、共享库各放一个
2. <strong>保持局部性</strong>：子目录配置只包含该模块相关的信息
3. <strong>避免重复</strong>：全局通用的内容放在根目录，子目录只放差异化配置
4. <strong>命名清晰</strong>：每个子目录 CLAUDE.md 顶部注释说明适用范围


#### 推荐的项目配置模板

````text
my-project/
├── CLAUDE.md                    # 根配置（<60行）
├── .claude/
│   └── rules/
│       ├── coding-style.md      # 通用编码风格（无 paths）
│       ├── error-handling.md    # 错误处理规范（paths: src/**/*.ts）
│       ├── testing.md           # 测试规范（paths: **/*.test.ts）
│       └── security.md          # 安全规范（paths: src/**/*.ts）
├── frontend/
│   ├── CLAUDE.md                # 前端专属配置
│   └── src/
├── backend/
│   ├── CLAUDE.md                # 后端专属配置
│   └── src/
└── docs/
    └── architecture.md          # 架构文档（@include 引用）
````

---


### 常见问题与陷阱


#### 常见问题

<strong>Q1: @include 引用的文件可以是非 Markdown 格式吗？</strong>

A: 不可以。被引用的文件必须是 Markdown（<code>.md</code>）格式。

<strong>Q2: 循环引用会被怎么处理？</strong>

A: Claude Code 内置循环检测机制。如果 A 引用 B、B 又引用 A（或更长的循环链），系统会检测到并跳过循环引用，不会导致无限递归或崩溃。但被跳过的内容不会生效，建议在编写时避免循环。

<strong>Q3: 子目录 CLAUDE.md 可以引用其他文件吗？</strong>

A: 可以。子目录的 CLAUDE.md 同样支持 <code>@include</code> 语法引用其他文件。

<strong>Q4: paths 中的路径是相对于哪个目录？</strong>

A: <code>.claude/rules/</code> 中的 paths 匹配相对于<strong>项目根目录</strong>。而 <code>@include</code> 中的路径相对于<strong>当前文件所在目录</strong>。

<strong>Q5: 可以同时使用三种机制吗？</strong>

A: 可以。它们互相独立、互补使用。推荐组合策略见第六节。


#### 常见陷阱

<strong>陷阱1：@include 不能节省 Token</strong>

很多用户误以为用 <code>@include</code> 拆分文件就能节省 Token。实际上 <code>@include</code> 只是启动时全量展开引用，所有内容仍然在上下文中。如果需要节省 Token，必须使用 <code>.claude/rules/</code> 条件规则或子目录 <code>CLAUDE.md</code>。

<strong>陷阱2：paths 路径写错导致规则不生效</strong>

paths 中的 glob 模式必须正确匹配目标文件路径。常见错误：

- 路径基准搞错（相对于项目根目录，不是相对于 rules 目录）
- 通配符使用不当（<code>&#42;.ts</code> 不匹配子目录，<code>&#42;&#42;/&#42;.ts</code> 才匹配所有层级）
- 排除规则顺序错误（排除规则必须放在包含规则之后）

<strong>陷阱3：子目录 CLAUDE.md 不会自动加载</strong>

子目录的 <code>CLAUDE.md</code> 不会在会话启动时加载，只有当 Claude 实际读取或修改该目录下的文件时才会触发。这意味着如果你只是让 Claude 看根目录的文件，子目录配置不会生效。

<strong>陷阱4：嵌套层级过深</strong>

虽然子目录 CLAUDE.md 支持无限嵌套，但过深的嵌套会导致配置难以维护。建议控制在 2-3 层以内。

<strong>陷阱5：@include 嵌套超过 5 层</strong>

<code>@include</code> 最多支持 5 层嵌套引用。如果超过这个深度，会报错。遇到这种情况应该改用 <code>.claude/rules/</code> 或子目录 <code>CLAUDE.md</code>。

<strong>陷阱6：YAML Frontmatter 格式错误</strong>

<code>.claude/rules/</code> 中的 YAML Frontmatter 必须严格遵循 YAML 语法：

- 用 <code>---</code> 包裹
- <code>paths</code> 字段是列表格式
- 缩进使用空格（不用 Tab）
- 文件路径用引号包裹（尤其包含特殊字符时）

---


### 十、参考资源

- <a href="https://docs.anthropic.com/">Anthropic 官方文档 - CLAUDE.md</a>
- <a href="https://github.com/anthropics/claude-code">Claude Code GitHub 仓库</a>
- <a href="https://github.com/isaacs/node-glob#readme">Glob 模式匹配文档</a>
- <a href="https://yaml.org/">YAML Frontmatter 规范</a>

---


## 02 - 会话中的记忆机制：六层模型


原讲义：[02 - 会话中的记忆机制：六层模型](https://heuqqdmbyk.feishu.cn/docx/Z5t8dKeB7oBre9xMmuEcdyBan4g)。

“六层模型”是原作者基于 Claude Code 2.1.252 的实现分析与教学划分，包含逆向分析所得的阈值、后台任务和模型调用细节。本笔记保留该分析，但未重新逆向验证；这些实现细节不等于所有 Claude Code 版本的固定保证，也不能直接套用到 ChatGPT 或 Codex。公开行为可对照 [Claude Code 官方记忆文档](https://code.claude.com/docs/en/memory)。


### Claude Code 记忆系统

各位同学好。这一节把会话中，claudecode用来负责记忆的模块讲一遍：包含了六个记忆模块分别在什么时候触发、干了什么、数据怎么处理的等知识点。

<strong>什么时候用？</strong>

1. 面试的时候
2. 自己设计智能体的记忆系统
3. 公司里面你要做技术分享了

都能直接拿来用。。。

这篇文章我写了整整三天，翻官方文档，又对着我自己电脑里面的claudecode做了逆向分析，每一步都留了能复现的验证方法。头发掉了一大堆，看我这么努力的份上，给兄弟我的课程点赞、收藏、投币、转发可好？

学习过程中有问题，兄弟们可以到我直播间吹牛逼，抖：尼古拉斯阿玮。不定期直播。

---

<strong>阅读声明：</strong>

1. <strong>阈值会变</strong>：下面文章里面的数字（24 小时、5 个会话、20000 token、200 行……）都是我在 Claude Code 2.1.252 上实测的，不是官方承诺的固定值，换个版本就可能不一样。所以每个数字我都标了它在源码里的位置，大家可以自己验。

> 阈值就是触发的边界值，最大值，最小值都算，比如：你给本课程投两个币，我就会回答你，兄弟/姐妹，牛逼！
> 
> 这个2，就是阈值的意思。

1. <strong>"六层记忆"是根据源码逆向分析得到的，不是官方申明。</strong> 官方的文档里没有六层记忆模型这个说法。真实源码里这六块是各自独立的模块，我按“作用域"、"存活多久"和执行顺序。把它们串成一个框架，方便理解，也方便面试表达。
2. <strong>官方说的"记忆系统"其实很窄。</strong>一般只指两块：CLAUDE.md 开发规则，和 Auto Memory 自动记忆。剩下几层官方没有公布出来，但它们确实在干记忆的活儿。
3. <strong>整条主线一句话：</strong>文指令记忆定规则，短期和工作记忆管当下，召回把记忆拉进当前回合，压缩防溢出，长期记忆沉经验，休眠重塑做保养。

六层里只有 ①④⑥ 能跨会话活下来，②③⑤ 会话一结束就没了。


#### 六层记忆模型总览

| <strong>层级</strong> | <strong>名称</strong> | <strong>作用域</strong> | <strong>载体</strong> | <strong>触发时机</strong> | <strong>生命周期</strong> |
| --- | --- | --- | --- | --- | --- |
| ① | 指令记忆 | 规则级 | <code>CLAUDE.md</code> 文件 + <code>rules/&#42;.md</code> | 会话启动加载；子目录按需注入 | 永久（手动维护） |
| ② | 短期记忆 | 会话级 | 内存中 <code>messages&#91;&#93;</code> 数组 | 每轮对话自动追加 | 会话结束即销毁 |
| ③ | 工作记忆 | 任务级 | 内存中工具状态/进度/游标 | 每轮对话动态更新 | 会话结束即销毁 |
| ④ | 长期记忆 | 持久级 | <code>memory/</code> &#43; <code>MEMORY.md</code> | 回合结束写入 | 永久（AutoDream 管理） |
| ⑤ | 压缩记忆 | 压缩级 | <code>messages&#91;&#93;</code> 内的摘要 + <code>compact&#95;boundary</code> 标记 | 上下文达水位 / 微压缩触发 | 会话内有效 |
| ⑥ | 休眠重塑 | 离线级 | 作用于第④层 <code>memory</code> 目录 | ≥24h 且 ≥5 次新会话 | 跨会话后台异步 |


#### 阶段零：记忆整理（Auto Dream）

这一层发生在<strong>会话开始之前</strong>，所以编成"阶段零"——它管的不是这次会话里发生的事，而是<strong>这次会话启动时能加载到什么</strong>。

触发条件：（<strong>两个必须同时满足</strong>，源码里面写 <code>{minHours:24,minSessions:5}</code>)

- 距上次整理 <strong>≥ 24 小时</strong>
- 期间新增 <strong>≥ 5 个会话</strong>（不算当前会话）

不满足则完全跳过。此外还有三重护栏：

| <strong>护栏</strong> | <strong>说明</strong> |
| --- | --- |
| 时间节流 | 两次检查之间至少间隔 <strong>10 分钟</strong>，不满足条件的会话不会反复去扫 |
| 文件锁 | 锁文件带持有者的PID，检测到进程还活着就直接跳过，防多开重复整理 |
| 失败回滚 | 整理失败会回滚，并把下次触发推迟到24小时之后，避免反复失败反复跑 |

它整理后的成果，会在<strong>下一次会话启动</strong>时被阶段一加载进来。

<strong>记忆整理开关:</strong>

1. tengu&#95;onyx&#95;plover：官方开关

    这是自动记忆的官方开关，有三个作用：

    1. 决定 AutoDream 能不能用
    2. <strong>获取那两个阈值</strong>，24 小时和 5 个会话就是装在它里面的
    3. 你什么都没设的时候，它就是默认状态
2. autoDreamEnabled：个人开关

    这是自动记忆的个人开关。前提是总闸先点头。

优先级是这样的（源码 <code>BUt()</code>）：

````text
第一道门：远程开关（总闸）enabled / available 
   │
   ├─ 没有 → 到此为止，直接 false
   │        （你 settings.json 里设了什么，连读都不读）
   │
   └─ 有   → 才轮到第二道门：你自己设的 autoDreamEnabled
              ├─ 设了 → 听你的
              └─ 没设 → 回落到远程开关自己的 enabled
````

要特别说一句：

<strong>配置文件覆盖不了总闸。</strong> 我们自己的配置文件只是第二道门上的一个闸把子——总闸不开，这道门压根没有意义。源码里第一句就是 <code>if(!Lln()) return !1</code>，短路返回，后面的用户设置没机会被执行到。

> <strong>结论：</strong>
> 
> 只有连接claudecode官方才能使用记忆的自动整理，否则就算在配置文件中配置了也无法使用。
> 
> <strong>解决方案：</strong>
> 
> 连官方风险很大，容易封号，可以每过一段段时间，用自然语言提示claudecode，主动提醒让他去整理即可。


#### 阶段一：会话启动加载

启动时系统扫描文件系统，按固定顺序收集并合并所有的 <code>CLAUDE.md</code>文件，随后加载长期记忆索引，最终组装出 System Prompt。


##### 加载开发宪法CLAUDE.md

源码函数 <code>dlr()</code> 里的压栈顺序就是<strong>注入 System Prompt 的顺序</strong>，自上而下：

| <strong>次序</strong> | <strong>作用域</strong> | <strong>路径（以 Windows 为例）</strong> | <strong>说明</strong> |
| --- | --- | --- | --- |
| 1 | 组织托管 | <code>C:&#92;Program Files&#92;ClaudeCode&#92;CLAUDE.md</code> | 企业下发的托管指令，用户不可改 |
| 2 | 组织托管 rules | <code>C:&#92;Program Files&#92;ClaudeCode&#92;.claude&#92;rules&#92;&#42;.md</code> | 托管规则 |
| 3 | 用户全局 | <code>&#126;/.claude/CLAUDE.md</code> | 跨所有项目生效 |
| 4 | 用户 rules | <code>&#126;/.claude/rules/&#42;.md</code> |  |
| 5 | 项目 | 从盘符根<strong>逐级向下到 cwd</strong>，每层取 <code>CLAUDE.md</code>、<code>.claude/CLAUDE.md</code>、<code>.claude/rules/&#42;.md</code> | 随 Git 提交、团队共享 |
| 6 | 项目本地 | 每层的 <code>CLAUDE.local.md</code> | 写入 <code>.gitignore</code>，不提交 |
| 7 | 自动记忆索引 | <code>memory/MEMORY.md</code> | 见下节 |
| 8 | 常驻记忆 | <code>AutoMemPinned</code> | 见下节 |

<strong>要注意的笑细节：</strong>

1. <strong>冲突解决规则：</strong>遵守就近原则
2. <strong>项目的单独模块下的CLAUDE.md 不在启动时加载。</strong> 它是你<strong>真的去读那个目录下的文件时</strong>，才会加载。
3. <strong>单文件有 4MB 字节上限</strong>，超限的 <code>CLAUDE.md</code> 会被直接跳过并记日志。在claudecode的原代码当中，先拿文件大小跟上限比，超了就直接放弃这个文件，内容一个字节都不进上下文。所以一个 5MB 的 CLAUDE.md 不是"读进来半截"，而是"根本没读"。日志里会留一条 <code>&#91;CLAUDE.md&#93; skipping &lt;路径&gt;: not a regular file or exceeds 4194304 byte limit</code>。

（判断用的是<strong>严格大于</strong>，所以正好 4194304 字节的文件仍然读得进来。）


> **勘误：**当前官方文档将 CLAUDE.md 描述为系统提示词之后提供的上下文／用户消息，并非自动提升为系统级指令。具体文件被读入的先后顺序，也不等于冲突规则一定得到确定性覆盖。


##### 加载memory.md索引文件

每次会被加载到上下文的记忆有几下几种：

1. <code>MEMORY.md</code> 索引文件，常驻上下文。

> 注：常驻上下文的意思是一定会被读出来，放在提示词当中交给大模型。

索引文件<strong>有硬截断</strong>：超过 <strong>200 行</strong>（<code>VD=200</code>）或 <strong>25000 字节</strong>（<code>VF=25000</code>）就按先到先停截断。索引必须一行一条、写短，写长了后面的条目根本进不来。

1. <strong>置顶记忆</strong>。记忆详细文件中在 yaml frontmatter 里写上<code>metadata.pinned: true</code> 

启动时会<strong>连正文一起</strong>注入，最多 4 条。这一层不靠召回，永远在场。

> <strong>结论：</strong>
> 
> 置顶记忆和AutoDream一样，都需要连接官方服务，否则无法正常使用。
> 
> <strong>解决方案：</strong>
> 
> 将长期记忆写在claude.md文件当中

源码中挑选的规则如下：

````text
e.filter(t=>t.pinnedState==="true")            // 筛出所有标了 pinned 的
 .toSorted((t,r)=>r.modifiedMs-t.modifiedMs)   // 按修改时间倒序
 .slice(0, Tqt)                                // 先取 8 条候选（Tqt=8）
// 把这 8 条的正文读出来，再取前 4 条真正注入
.slice(0, G0)                                  // G0=4
````

所以它不是"按文件名取前 4 个"，er是<strong>按修改时间取最新的 4 个</strong>。我们如果标了 6 条，那么第 5、6 条会被读出来、但不会进入上下文。这就是为什么多标确不生效的原因。

<strong>为什么要有置顶记忆？</strong>

因为召回是"每轮挑 5 条最相关的"，天生带运气成分——重要的东西也可能这一次没被挑中。

就比如：像"这个项目用 pnpm 不许用 npm""回答一律用中文"这种每次都必须知道的，就该置顶。

> 这两个数字最容易串，记住这组对照：<strong>召回 5 条，置顶 4 条。</strong>

此时上下文已包含以下四个内容：

- 多个<code>CLAUDE.md</code>规则文件
- rules文件夹中的补充说明
- 记忆索引memory.md
- 置顶记忆。

此时还没有召回记忆，因为召回记忆是跟用户说的话有关的，现在阶段一只是准备工作


#### 阶段二：每轮对话执行循环

每轮循环：

````text
1 用户输入 → 追加到 messages[]（短期记忆）
2 语义召回 Relevant Memory Recall（见下）
3 微压缩 Micro-Compaction（按需，见下）
4 组装上下文：System + messages[] + 召回记忆
5 调用主模型，获取响应
6 执行工具调用（读/写文件、bash 等）
7 工具结果追加 messages[]，并实时落盘 Transcript
8 更新工作记忆（进度/偏移/重试状态）
9 返回结果，等待下一轮
````


##### 短期记忆与工作记忆

<strong>短期记忆</strong>：

就是内存里那个 <code>messages&#91;&#93;</code> 数组，每轮往里追加 user、assistant 和工具消息，会话一结束就销毁。模型靠上下文窗口"看到"全部历史。

<strong>工作记忆</strong>：

不是对话文本，是任务执行状态：做到哪一步了、哪个文件改过、哪个测试跑过、工具重试到第几次、记忆提取的游标推到哪了。它的作用是让长任务断了之后能从断点接着干，而不是从头再来。


##### 记忆召回（每轮触发）

这是模型"想起"具体记忆的唯一方式。

MEMORY.md 只是目录，记忆召回才是"翻到具体那一页"。

它不是启动时做一次，而是每一轮用户输入都会执行。

流程如下：

````text
用户输入
→ 先在内存索引里检索（倒排索引 + 相关度打分）
   索引范围：最多 200 个文件、单文件不超过 1MB，MEMORY.md 本体排除在外
   索引不可用就回落到扫文件：最多 200 个 md（Pqt=200），读 frontmatter 按修改时间倒序
→ 把已经展示过的记忆过滤掉
→ 按相关度排序，取前 5 条
→ 把这 5 条交给 Sonnet 做语义选择
→ 选中的文件读完整正文，注入当前回合
→ 拼成完整上下文，调主模型
````

<strong>两种召回模式</strong>：

<code>memory&#95;recall</code> 系统消息带 <code>mode</code> 字段，取值为

- <code>select</code>：返回被选中的完整文件正文（默认走这条）
- <code>synthesize</code>：由 Sonnet 把大量零碎记忆<strong>蒸馏成一段话</strong>再注入

<strong>选择器用哪个模型？</strong>

<strong>以下优先级</strong>：

参考源码<code>up（）</code>

1. 配置文件的 <code>ANTHROPIC&#95;DEFAULT&#95;SONNET&#95;MODEL</code>

    如果设置就发送这个个模型，跟 Anthropic 的 Sonnet 没关系
2. 如果没有设置，用程序内置的 sonnet 档位<code>claude-sonnet-5</code>
3. 再兜底 <code>sonnet46</code>，也就是 <code>claude-sonnet-4-6</code>。

一句话：<strong>这一步发出去的是"你配置里 sonnet 档位对应的那个模型"，不是"Anthropic 的 Sonnet"。</strong>

> 注：
> 
> 所以要是<strong>只配了 </strong><strong><code>ANTHROPIC&#95;BASE&#95;URL</code></strong><strong> 指向三方、没配 </strong><strong><code>ANTHROPIC&#95;DEFAULT&#95;SONNET&#95;MODEL</code></strong>：模型名会落成 <code>claude-sonnet-4-6</code>（第 3 段那个值），<strong>但这个请求照样发往你配的 </strong><strong><code>ANTHROPIC&#95;BASE&#95;URL</code></strong>——也就是发给三方。三方不认这个型号名，多半直接报错。
> 
> <strong>这就是接三方时 </strong><strong><code>ANTHROPIC&#95;BASE&#95;URL</code></strong><strong> 和 </strong><strong><code>ANTHROPIC&#95;DEFAULT&#95;SONNET&#95;MODEL</code></strong><strong> 必须成对配的原因。</strong>

<strong>相关度地板</strong>：

还有一道相关度下限（源码里叫 relevance floor，直译"相关度地板"）。检索出来的分数有个下限，分数不够就宁可不召回。如果零召回有以下几种原因：

- <code>below&#95;relevance&#95;floor</code>（全在地板下）
- <code>no&#95;servable&#95;hits</code>（无可用命中）
- <code>all&#95;filtered</code>（全被过滤）
- <code>index&#95;sweep&#95;incomplete</code>（索引未扫完）
- <code>no&#95;candidates</code>（无候选）。

这也是"为什么有时候记忆没被想起来"的主要原因。


> **补充：**“每轮必调 Sonnet、最多五条”等是原作者对特定版本和开关状态的分析。当前官方文档描述的是启动时加载 MEMORY.md 的前 200 行或 25KB，详细记忆按需用文件工具读取，不能把某一种内部实现当成所有版本的固定流程。


##### 微压缩 Micro-Compaction（不是每轮都做）

很多人以为每轮调模型前都会清理一遍旧内容，实际不是。真实机制是跟模型配合的。（

> 注：就是配置文件里 <code>ANTHROPIC&#95;BASE&#95;URL</code>指向的地址

在微压缩的时候会进行以下判断：

1. claudecode先估算：这批旧工具输出清掉能省多少 token，<strong>省不到 20000 token（</strong><strong><code>Iln=20000</code></strong><strong>）就不参与</strong>；
2. 够格了，就在正常的模型请求里带一个 <code>context&#95;hint</code> 参数，相当于报备"我有清理能力"
3. 配置的那个大模型那边上下文吃紧回压之后，claudecode才真正动手，时机是 <code>onRequestError</code> 和 <code>onStreamFallback</code>（返回错误、流式回退）这两个地方，日志长这样：<code>&#91;KEEP-RECENT MC&#93; context&#95;hint trigger, cleared N tool results, kept last M</code>；
4. 清理方式是保留最近若干条工具结果，更早的<strong>替换成一个固定字符串</strong> <code>&#91;Old tool result content cleared&#93;</code>，同时在会话里打一个隐藏标记 <code>microcompact&#95;boundary</code>。

注意：

全程不调模型：第一步是纯算数，第三步听信号，第四步是字符串替换。

它是"日常保洁"。与后面说的全量压缩"大扫除"配合。

每轮微压缩和自动全量压缩的对比：

| <strong>维度</strong> | <strong>微压缩</strong> | <strong>全量压缩（autoCompact）</strong> |
| --- | --- | --- |
| 触发频率 | 每轮都执行 | Token 达 &#126;92% 才触发 |
| 收益门槛 | 预估可省 ≥ 20000 token 才参与 | 无（到水位就压） |
| 压缩对象 | 旧工具输出 | 整段对话历史 |
| 是否调用模型 | 否（规则清理） | 是（生成摘要） |
| 成本 | 零 | 额外消耗 token |
| 信息损失 | 低（只丢旧工具输出） | 高（整段历史被替换） |


> **勘误：**本节标题和正文说按条件触发，而对比表写“每轮都执行”，两者不一致。应区分“检查是否需要压缩”和“实际删除旧工具内容”；也不能用固定 92% 代替后文带窗口大小、缓冲量的公式。


##### Transcript 实时记录

内存里跑 <code>messages&#91;&#93;</code> 的同时，每轮都会往磁盘追加写一份，路径跟记忆目录是同一个父目录，

比如说：

> 地址为：&#126;/.claude/projects/&lt;项目名&gt;/&lt;会话-id&gt;.jsonl
> 
> 就是我们上课讲的聊天历史记录文件

我们的输入、AI 的回复、工具调用和结果，逐条记下来。

它有三个用处：

- AutoDream 搜数据的来源
- 事后审计和调试的依据
- 会话恢复的依据。

压缩出来的摘要也会以 <code>compact&#95;boundary</code> 的形式留在里面。


##### Prompt Cache 共享

主会话的 System Prompt 和前几轮已经在服务端缓存了，fork 出去的子 Agent 直接命中缓存，不用重新处理这些 token。


#### 阶段三：上下文管理与压缩

这一节讲的是第⑤层，也就是"防溢出"到底怎么防。

<strong>水位线不是固定百分比，</strong>下面是在源码里面的真实算法：

````text
自动压缩阈值 = min(窗口 − 窗口 × 缓冲比例, 窗口 − 13000)
告警水位     = 阈值 − 20000
硬阻塞水位   = 窗口 − 3000
````

注意，<strong>阈值是"窗口减 13000 token"这种预留式算法，不是一个固定百分比</strong>。你可能听过"到 92% 压缩"的说法，那只是某个具体窗口尺寸下的折算值——20 万窗口算下来是 <code>(200000−13000)/200000 ≈ 93.5%</code>。讲的时候把窗口口径一起说，不然换个模型就对不上。

<strong>这几个都能改：</strong>

| 名称 | 作用 |
| --- | --- |
| <code>CLAUDE&#95;AUTOCOMPACT&#95;PCT&#95;OVERRIDE</code> | 强制按百分比设阈值 |
| <code>CLAUDE&#95;CODE&#95;BLOCKING&#95;LIMIT&#95;OVERRIDE</code> | 覆盖硬阻塞水位 |
| <code>autoCompactWindow</code> / <code>CLAUDE&#95;CODE&#95;AUTO&#95;COMPACT&#95;WINDOW</code> | 设置自动压缩的窗口大小 |
| <code>autoCompactEnabled</code> / <code>DISABLE&#95;AUTO&#95;COMPACT</code> / <code>DISABLE&#95;COMPACT</code> | 关掉自动压缩 |


##### 全量压缩做什么

到水位之后，系统会 fork 一个子 Agent 生成对话摘要，然后把整段历史换成这段摘要，留下一个 <code>compact&#95;boundary</code> 标记，界面上看起来就是"会话被折叠了"。触发来源分两种：自动到水位（<code>auto</code>），和手动敲 <code>/compact</code>。

<strong>压缩摘要的固定模板（9 个 Section，2.1.252 原文）：</strong>

| <strong>序号</strong> | <strong>Section</strong> | <strong>含义</strong> |
| --- | --- | --- |
| 1 | Primary Request and Intent | 用户所有显式请求与意图 |
| 2 | Key Technical Concepts | 涉及的技术概念、框架 |
| 3 | Files and Code Sections | 涉及的文件与代码片段，及其重要性 |
| 4 | Errors and fixes | 遇到哪些错误、如何修复、用户给出的纠正 |
| 5 | Problem Solving | 已解决的问题与进行中的排查 |
| 6 | All user messages | <strong>全部</strong>用户消息（非工具结果），安全相关约束要求逐字保留 |
| 7 | Pending Tasks | 用户明确交代但尚未完成的任务 |
| 8 | Current Work | 压缩发生前正在做的事，含文件名与代码片段 |
| 9 | Optional Next Step | 下一步动作，且必须与用户最近的显式请求直接对齐 |

这份模板本身就是"怎么写出好的上下文摘要"的范本，四个要点：<strong>保留原话、保留错误、保留没做完的、下一步要引用用户原话</strong>。照着这四条写，压缩之后模型基本不会跑偏。


#### 阶段四：长期记忆写入与沉淀

这是唯一"跨会话持久化"的动态记忆层，就是我们课堂讲的自动记忆。

位置： C盘下的/.claude/projects/&lt;project&gt;/memory/。


##### 存储结构与四种类型

````text
MEMORY.md             # 索引，≤200 行 / ≤25KB（硬截断，见阶段一）
user_role.md          # type: user
feedback_testing.md   # type: feedback
project_auth.md       # type: project
reference_linear.md   # type: reference
````

<strong>四种类型：</strong>

- user（用户身份/偏好）
- feedback（用户反馈）
- project（项目事实/决策）
- reference（外部系统指针）

写之前有个筛选标准：<strong>只记"跨会话还有价值、而且从代码里推不出来"的信息</strong>。代码模式、架构分析这类能自证的东西明确不记。


##### 三个保护机制

- <strong>互斥保护：</strong>

主 Agent 这一轮已经手动写过记忆了，后台提取就主动跳过这一轮，防止重复写。

> 注：大家如果要看日志，日志是 <code>&#91;extractMemories&#93; skipping — conversation already wrote to memory files</code>。

- <strong>背压处理</strong>：

    这词听着唬人，其实就是"上一批活儿还没干完，新的又来了怎么办"。这里的处理方式是<strong>暂存合并</strong>：

    上一轮提取还没跑完，新请求又来了，系统把上下文存起来，等这轮忙完再合并跑一次，而不是并发堆一堆。。

> 注：大家如果要看日志，日志是（ <code>stashing for trailing run</code>）

- <strong>权限沙箱</strong>：

    Fork 子 Agent 仅拥有 memory 目录的读写权限，禁止访问项目源码与 Bash、禁止联网。

    下面是我从源码里面扒出来的权限：

````text
原文：
Only read-only shell commands and rm (no flags except -f) of .md files under <memoryDir> ... are permitted
翻译:
仅允许读取 `<memoryDir>` 下 `.md` 文件的只读 shell 命令和 rm（除 `-f` 以外的其他参数）
````


> **勘误：**引文 `rm (no flags except -f)` 的意思是“rm 只允许使用 -f，不能带其他选项”，原中文翻译反了；同一引文还明确允许特定只读 shell 命令，与“禁止 Bash”的概括并不完全一致。这段描述只用于理解原作者分析的权限范围，不是清理文件的操作指令。


##### 用户反馈记忆

源码明确要求正负反馈都要记。只记批评不记表扬，时间长了模型会过度谨慎、事事请示、效率下降。

| <strong>反馈方向</strong> | <strong>示例</strong> | <strong>不记的后果</strong> |
| --- | --- | --- |
| 负反馈（纠正） | 不要 mock 数据库 | 模型重复犯错 |
| 正反馈（确认） | 对，就这样写测试 | 模型趋向保守，不敢决策 |


##### 记忆是提示，不是真理

记忆是提示不是真理。注入上下文时系统同时会下达一段警告：

> 2.1.252 原文意译：
> 
> 出现在 <code>&lt;system-reminder&gt;</code> 里的召回记忆是背景信息，反映的是<strong>写下时的</strong>状态。若它提到某个文件、函数或开关，<strong>引用前必须先验证它现在还存在</strong>。

配套的工程习惯：

- 引用记忆里的文件路径前，先 <code>ls</code> 确认还在；
- 引用函数签名前，先 <code>grep</code> 确认没被改；
- 当记忆与当前代码冲突时，<strong>优先信任实时文件系统状态</strong>。


#### 阶段五：会话结束的清理与保留

| 对象 | 所属层 | 结果 |
| --- | --- | --- |
| <code>messages&#91;&#93;</code> 数组 | ② 短期记忆 | 销毁，不保留 |
| 工作记忆（工具状态/进度） | ③ 工作记忆 | 清空，不保留 |
| 压缩摘要 | ⑤ 压缩记忆 | 会话内有效；<strong>不再跨会话单独保留</strong> |
| Transcript <code>.jsonl</code> | 旁路落盘 | <strong>保留</strong>（审计 / 恢复 / AutoDream 数据源） |
| <code>memory/&#42;.md</code> &#43; <code>MEMORY.md</code> | ④ 长期记忆 | 保留，永久，供所有会话复用 |
| 所有<code>CLAUDE.md</code> 文件 | ① 指令记忆 | 保留，永久 |


> **补充：**内存状态结束与磁盘记录删除是两件事。Transcript 可能支持恢复，也受保留期设置影响；memory 文件则留到被编辑或删除。原文中的“永久”“会话结束就没了”应结合具体载体理解，不代表所有历史信息都会立即消失或无限保留。


### 附一：数据发送给谁？

很多同学觉得，我连了三方模型，DeepSeek或者其他，就跟claudecode官方Anthropic没关系了，其实不是这样的，claudecode的源码中很多地方写死了会访问Anthropic的官方服务，这就是为什么明明是同一个工具，连接官方模型和连接三方模型效果不一样的原因。

什么时候访问官方服务，什么时候访问模型其实只有两种情况：

- <strong>跑模型</strong> → 发给<strong>你配置文件里配的那个模型</strong>。接第三方就是第三方。
- <strong>取开关、上报</strong> → 发给 <strong>Anthropic 的官方服务</strong>。这一路<strong>地址写死在代码里</strong>，你换不了，也改不成发给三方。


#### 第一类：四个环节各用什么模型

这块最容易混，一张表说清：

| <strong>环节</strong> | <strong>用的模型</strong> |
| --- | --- |
| 记忆召回选择 | Sonnet 档位 |
| 记忆提取子 Agent | Fork 主会话、复用缓存，最多5轮 |
| 全量压缩摘要 | 主对话模型，不是 sonnet，只有 1 轮 |
| 微压缩 | 不调模型，纯规则替换占位符 |


##### 第二类：Anthropic 官方服务的清单

| <strong>用途</strong> | <strong>地址</strong> | <strong>能不能关</strong> |
| --- | --- | --- |
| 远程开关 | <a href="https://api.anthropic.com/">https://api.anthropic.com/</a>，写死，请求还要带认证头 | 拿不到就取默认值，可以在配置文件中禁用 |
| 事件上报 | <a href="https://api.anthropic.com/api/event_logging/v2/batch">https://api.anthropic.com/api/event&#95;logging/v2/batch</a>，200 条一批、超时 10 秒 | DISABLE&#95;TELEMETRY / DO&#95;NOT&#95;TRACK |
| CLI版本错误上报 | <a href="https://browser-intake-us5-datadoghq.com/api/v2/logs">https://browser-intake-us5-datadoghq.com/api/v2/logs</a> | DISABLE&#95;ERROR&#95;REPORTING |
| 桌面版本错误上报 | &#42;.sentry.io、o1158394.ingest.us.sentry.io | 客户端有开关 |
| 更新检查 | <a href="http://releases.claude.com">releases.claude.com</a> | 客户端有开关 |
| 登录 | <a href="http://claude.ai">claude.ai</a>、<a href="http://platform.claude.com">platform.claude.com</a> | — |
| 谷歌认证 | <a href="http://oauth2.googleapis.com">oauth2.googleapis.com</a>、<a href="http://sts.googleapis.com">sts.googleapis.com</a>、<a href="http://iamcredentials.googleapis.com">iamcredentials.googleapis.com</a> | 不用就不连 |

<strong>注意点：</strong>

<strong>事件上报和远程开关的地址是写死的 </strong><strong><code>api.anthropic.com</code></strong><strong>，跟配置文件无关</strong>

源码如下：

````text
// 遥测 endpoint 是这么拼出来的
let t = e.baseUrl || (process.env.ANTHROPIC_BASE_URL === "https://api-staging.anthropic.com"
                        ? "https://api-staging.anthropic.com"
                        : "https://api.anthropic.com");   // ← 只有 staging 是特例
this.endpoint = ${t}${e.path || "/api/event_logging/v2/batch"};
this.timeout = e.timeout || 1e4;              // 10 秒
this.maxBatchSize = e.maxBatchSize || 200;    // 200 条一批
````

> <strong>咱们把模型接到第三方，事件上报、开关请求照样往 Anthropic 官方发</strong>，不会跟着跑到三方去。
> 
> 反过来——<strong>三方那边永远拿不到你的开关</strong>，这就是 AutoDream 在第三方环境下跑不起来的根本原因（详见阶段零）。


##### 三条结论

1. <strong>没有"记忆服务器"这回事</strong>

    六层里全是本地内存和本地文件，长期记忆落在本地 <code>memory/</code> 目录。联网只为了两件事——<strong>跑模型</strong>和<strong>取开关</strong>。微压缩那个 <code>context&#95;hint</code> 是跟着模型请求走的 beta 参数，不是独立服务。
2. <strong>开关是"云端求值 + 按组织灰度"</strong>

 判定在云端算好再下发，缓存键带 <code>organizationUUID</code>。所以<strong>同一个版本、不同账号看到的开关可以不一样</strong>，这就是为什么"别人有、我没有"的原因。

1. <strong>断网也能用</strong>

     <code>CLAUDE.md</code>、<code>memory/</code>、transcript 全是本地文件。<strong>只有模型调用离不开网。</strong>


### 附二：全流程时间轴

六个阶段按真实时间顺序串起来，如下图所示：

````text
[上一个会话结束]
      │
      ├─ 写 transcript.jsonl，保留
      │
      └─ 满足"≥24 小时 + ≥5 个新会话" → AutoDream 在后台整理 memory/

[这次会话启动]
      │
      ├─ 扫描合并 CLAUDE.md 家族和规则（托管 → 用户 → 项目 → 本地）
      ├─ 常驻 MEMORY.md 索引（200 行 / 25KB 以内）+ 常驻记忆正文
      │
      └─ 进入每轮循环
            ├─ 输入追加进 messages[]
            ├─ 召回：索引检索 → 过滤 → 取 5 条 → Sonnet 选择 → 注入
            ├─ 微压缩：API 回压时，能省 20000 token 以上才做
            ├─ 调主模型 → 工具调用 → 结果入 messages[] 并落盘
            ├─ 水位检查 → 到线就全量压缩，出 9 段摘要 + compact_boundary
            └─ 回合结束 → 主 Agent 直接写 memory/
                        └─ 开关打开且本轮没写过 → fork 子 Agent 提取

[这次会话结束]
      └─ 销毁 messages[] 和工作记忆，保留 transcript、memory/、CLAUDE.md
````


### 附三：自己逆向的方式

下面跟同学分享逆向安装包的方式，但是这是逆向工程里面最简单的方式：提取明文静态分析。

我们并没有破坏任何东西，只是找到源码在哪，并验证我们的结论而已。

````text
# 1. 找到程序，先记版本号
cd "$(dirname "$(which claude)")/node_modules/@anthropic-ai/claude-code"
cat package.json | grep version

# 2. 去掉夹杂的空字节，方便一次 grep 到字符串表里的内容
#    JS 本身已经是明文，不加这步也能搜，只是命中会碎
tr -d '\000' < bin/claude.exe > /tmp/cc.js

# 3. 想查什么就搜什么
grep -o '.\{80\}session-memory.\{80\}' /tmp/cc.js
grep -o 'pEt={minHours:24,minSessions:5}' /tmp/cc.js
grep -o '.\{60\}Primary Request and Intent.\{900\}' /tmp/cc.js
````

<strong>注意点</strong>：

1. 每条结论先记版本号
2. 找到函数名就顺着读调用方，别只看常量
3. 搜不到就写"未复现"，不要脑补成"官方就是这样"
4. Claude Code 是闭源商业软件，我们现在只是分析<strong>自己机器上合法安装的程序</strong>，性质是属于学习研究


### 附四：各个节点的阈值（2.1.252）

| <strong>项</strong> | <strong>值</strong> | <strong>符号/设置</strong> |
| --- | --- | --- |
| AutoDream 触发 | ≥24h 且 ≥5 个新会话 | <code>pEt</code> |
| AutoDream 检查节流 | 10 分钟 | <code>fUn=600000</code> |
| <code>MEMORY.md</code> 索引上限 | 200 行 / 25000 字节 | <code>VD</code> / <code>VF</code> |
| <code>CLAUDE.md</code> 单文件上限 | 4 MB | <code>ele=4194304</code> |
| Pinned 记忆常驻条数 | 4 条（候选池 8） | <code>G0</code> / <code>Tqt</code> |
| 召回候选上限 | 200 个文件（扫描路径） | <code>Pqt=200</code> |
| 召回注入条数 | 最多 5 条 | <code>.slice(0,5)</code> |
| 记忆索引范围 | 2000 文件 / 单文件 1MB，排除 <code>MEMORY.md</code> | 索引扫描配置 |
| 微压缩收益门槛 | 20000 token | <code>Iln=20000</code> |
| 召回选择器模型 | Sonnet 档位（env → 配置 sonnet → 兜底 4-6） | <code>up()</code> / <code>Fs()</code> |
| 记忆提取子 Agent 轮数 | 5 | <code>maxTurns:5</code> |
| 压缩摘要子 Agent 轮数 | 1 | <code>maxTurns:1</code> |
| 自动压缩阈值 | 窗口 − 13000 | <code>E9()</code> |
| 告警水位 | 阈值 − 20000 | <code>Qge()</code> |
| 硬阻塞水位 | 窗口 − 3000 | <code>Qge()</code> |


> **勘误：**正文记忆索引范围写“200 个文件”，本表写“2000 文件”，无法仅凭这份讲义判定正确值。保留两个原始说法，实际复现时需核对对应版本、索引路径与回退扫描路径，不混为同一项限制。


### 附五：专业术语表

1. 主Agent

    大家就可以理解为现在跟我们对话的claudecode就是主Agent
2. 子Agent

    可以理解为主Agent的分身，后台自动运行的时候。
3. Fork 子Agent

<strong>Fork ≠ 生成。</strong> fork来自于git，意思是"<strong>另起一条线程、跑完不污染主线</strong>"。

<strong>它到底做了什么</strong>

子Agent启动时，把主会话的那份 <code>messages</code> 作为自己的<strong>初始上下文</strong>塞进去。所以子代理是"<strong>带着主会话的记忆去干活</strong>"，不是凭空生成一个新东西。干完只把<strong>结果</strong>交回来，中间过程不进主线对话。

<strong>为什么必须 fork</strong>

提取记忆、写压缩摘要这类活儿，如果直接在主线里跑，模型会在 <code>messages&#91;&#93;</code> 里看到"自己跟自己开会"，把对话流搞乱。fork 让这些后台活儿另起一条线——<strong>主线干净，结果回流</strong>。

> <strong>课堂一句话</strong>：生成是无中生有造内容；<strong>fork 是拿我现在的上下文复制一份，另起一条执行线干活，干完只把结果交回来。</strong>


> **勘误：**fork 一词并非源自 Git，在操作系统等领域早已有“分叉／复制执行上下文”的含义。此处可理解为从父会话上下文派生子任务，是否复制全部历史、共享哪些状态，取决于具体实现。


## 03 - 提示词工程（进阶版）


原讲义：[03 - 提示词工程（进阶版）](https://heuqqdmbyk.feishu.cn/docx/VvvmdNa4YoLE0BxNaQacxkh4n3f)。

本部分保留原讲义引用的 OpenAI API 参数、模型名称与时间表。整理时 OpenAI 官方开发文档返回 403，因此其中关于 GPT-6 Astra、GPT-5.6、部分推理配置项以及 v1/prompts 的 2026-11-30 日期等具体说法，未能独立核实。学习概念时可按原文对照；实际编程前，应以账号可用模型及对应接口文档为准。原示例的语法问题在相关小节注明。


> <strong>适用：</strong>开发者、需要写生产级提示词的人<br><strong>前置：</strong>建议先读《零基础版》了解基本写法
> 
> 1. 《Prompt engineering》提示词工程指南
> 
> https://developers.openai.com/api/docs/guides/prompt-engineering
> 
> 2. 《Using GPT-6 Astra》GPT-6 Astra 使用指南
> 
> https://developers.openai.com/api/docs/guides/latest-model
> 
> 3. 《Reasoning best practices》推理模型最佳实践
> 
> https://developers.openai.com/api/docs/guides/reasoning-best-practices
> 
> 4. 《Reasoning models》推理模型与参数详解
> 
> https://developers.openai.com/api/docs/guides/reasoning
> 
> 5. 《Structured Outputs》结构化输出
> 
> https://developers.openai.com/api/docs/guides/structured-outputs
> 
> 6. 《Function calling》函数调用
> 
> https://developers.openai.com/api/docs/guides/function-calling
> 
> 7. 《Text generation》文本生成
> 
> https://developers.openai.com/api/docs/guides/text
> 
> 8. 《Evals》模型评估指南
> 
> https://developers.openai.com/api/docs/guides/evals
> 
> 9. 《Create a response》Responses API 接口参考
> 
> https://developers.openai.com/api/reference/resources/responses/methods/create
> 
> 1. 《GPT-5 Troubleshooting Guide》GPT-5 排错指南
> 
> https://developers.openai.com/cookbook/examples/gpt-5/gpt-5&#95;troubleshooting&#95;guide
> 
> 1. 《Conversation state》会话状态管理
> 
> <a href="https://developers.openai.com/api/docs/guides/conversation-state">https://developers.openai.com/api/docs/guides/conversation-state</a>
> 
> 1. 《llms.txt》OpenAI 开发者文档全站索引
> 
> <a href="https://developers.openai.com/llms.txt">https://developers.openai.com/llms.txt</a>

---


### 一、消息角色与指令层级

<strong>这节讲什么：</strong>

API 里的指令是<strong>分等级</strong>的。你(开发者)写的规则和用户输入的话如果打架，模型按固定顺序决定听谁的——用户翻不掉你设的规则。<strong>做产品的人必看，只写提示词的可以跳过。</strong>


#### 这个参数在哪、怎么传

<code>instructions</code> <strong>不是聊天框里的设置项</strong>，它是 <strong>Responses API 请求体里的一个顶层字段</strong>——和 <code>model</code>、<code>input</code> 平级，由写代码的人在调用时传进去:

````text
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-astra"，                             # 选模型
    instructions="Talk like a pirate."，              # ← 你的规则，顶层参数
    input="Are semicolons optional in JavaScript?"，  # ← 用户的话
)

print(response.output_text)
````

换成裸 HTTP 看最清楚——参数就摆在 JSON 第一层：

````text
curl https://api.openai.com/v1/responses \
    -H "Content-Type: application/json" \
    -H "Authorization: Bearer $OPENAI_API_KEY" \
    -d '{
        "model": "gpt-6-astra"，
        "instructions": "Talk like a pirate."，
        "input": "Are semicolons optional in JavaScript?"
    }'
````


> **勘误：**本篇多段 Python、JSON、curl 示例在结构分隔处使用了中文逗号 `，`，不能直接执行或解析。实际使用时将语法分隔符改为英文逗号 `,`，保留字符串内部正常的中文标点；Python 的布尔值是 `True/False`，JSON 中是 `true/false`。原示例仍完整保留，便于对照。


#### <code>instructions</code> 的定义

OpenAI 定义:

> The <code>instructions</code> parameter gives the model high-level instructions on how it should behave while generating a response， including tone， goals， and examples of correct responses. <strong>Any instructions provided this way will take priority over a prompt in the </strong><strong><code>input</code></strong><strong> parameter.</strong>
> 
> ——<code>instructions</code> 给出的是<strong>高层行为指令</strong>(语气、目标、正确响应示例);<strong>其优先级高于 </strong><strong><code>input</code></strong><strong> 里的提示词。</strong>

<strong>一个容易被忽略的坑</strong>:<code>instructions</code> <strong>只对当前这次生成请求生效</strong>。如果你用 <code>previous&#95;response&#95;id</code> 管理会话状态，<strong>之前轮次用过的 </strong><strong><code>instructions</code></strong><strong> 不会保留在上下文里</strong>。


#### 三种角色

这三种角色<strong>都写在 </strong><strong><code>input</code></strong><strong> 数组里</strong>，每项用 <code>role</code> 字段标明身份：

| <strong>角色</strong> | <strong>含义</strong> | <strong>优先级</strong> | <strong>谁来写</strong> |
| --- | --- | --- | --- |
| <code>developer</code> | <strong>应用开发者</strong>提供的指令 | 排在 <code>user</code> <strong>之前</strong> | 你(写代码的人) |
| <code>user</code> | <strong>最终用户</strong>提供的指令 | 排在 <code>developer</code> <strong>之后</strong> | 你的用户 |
| <code>assistant</code> | <strong>模型生成</strong>的消息 | —— | 模型(你回填进历史) |

<strong>一次完整请求里，三种角色写出来是这样</strong>:

````text
response = client.responses.create(
    model="gpt-6-astra"，
    input=[
        # ① developer:你定的规则 —— 人格、业务边界、输出格式
        {"role": "developer"， "content": "你是客服助理，只回答本店商品问题，不评价竞品。"}，

        # ② user:用户上一轮说的话
        {"role": "user"， "content": "你们家这件衣服质量怎么样?"}，

        # ③ assistant:模型上一轮的回复，由你回填进历史
        {"role": "assistant"， "content": "面料是新疆长绒棉，支持 30 天无理由退换。"}，

        # ② user:用户这一轮的新问题
        {"role": "user"， "content": "和隔壁家的比怎么样?"}，
    ]，
)

print(response.output_text)
````

<strong><code>assistant</code></strong><strong> 消息长什么样</strong>:它不是你手写的文本，而是<strong>模型返回的 </strong><strong><code>output</code></strong><strong> 数组里的一项</strong>，结构固定:

````text
[
  {
    "id": "msg_67b73f697ba4819183a15cc17d011509"，
    "type": "message"，
    "role": "assistant"，
    "content": [
      {
        "type": "output_text"，
        "text": "面料是新疆长绒棉，支持 30 天无理由退换。"，
        "annotations": []
      }
    ]
  }
]
````

多轮对话时，把模型的输出连同新的 <code>user</code> 消息一起放回 <code>input</code>，模型才有上下文。<strong><code>output</code></strong><strong> 数组常常不止一项</strong>(可能夹着工具调用、推理 token 信息)，别以为文字就一定在 <code>output&#91;0&#93;.content&#91;0&#93;.text</code>——用 SDK 的 <code>response.output&#95;text</code> 取值最省事。

OpenAI「手动管理会话状态」页给的多轮例子，就是这个格式:

````text
response = client.responses.create(
    model="gpt-6-astra"，
    input=[
        {"role": "user"， "content": "knock knock."}，
        {"role": "assistant"， "content": "Who's there?"}，
        {"role": "user"， "content": "Orange."}，
    ]，
)
````

OpenAI 给了一个非常好记的类比：

> You could think about <code>developer</code> and <code>user</code> messages like <strong>a function and its arguments</strong> in a programming language.
> 
> - <code>developer</code> messages provide the system's rules and business logic， <strong>like a function definition</strong>.
> - <code>user</code> messages provide inputs and configuration to which the <code>developer</code> message instructions are applied， <strong>like arguments to a function</strong>.
> 
> ——把 developer 消息想成<strong>函数定义</strong>，user 消息想成<strong>传给函数的参数</strong>。
> 
> <strong>注</strong>:推理模型自 <code>o1-2024-12-17</code> 起支持 <strong>developer 消息而非 system 消息</strong>，以对齐模型的指令链行为。


> **补充：**开发者指令高于用户指令是指令处理规则，不能替代程序的权限检查、输入验证和数据隔离。`system`、`developer`、`instructions` 的支持范围要按所用模型与接口确认，不应从某个历史模型推广到所有模型。`assistant` 历史也可以按 API 接受的格式回填，但不能把编造的回复当作真实业务结果。


#### 什么时候用哪个

<strong>同一个需求，两种写法都能实现</strong>:

| <strong>写法</strong> | <strong>长什么样</strong> | <strong>特点</strong> |
| --- | --- | --- |
| <code>instructions</code> 参数 | <code>instructions="Talk like a pirate."</code> | 简洁;<strong>只对当前这一次请求生效</strong> |
| <code>developer</code> 消息 | <code>input=&#91;{"role": "developer"， "content": "Talk like a pirate."}， …&#93;</code> | 写进对话流，可随历史一起留存 |

<strong>按用途选</strong>:

| <strong>你要做的事</strong> | <strong>用什么</strong> |
| --- | --- |
| 定全局规则(人格、语气、业务边界) | <code>instructions</code> 参数 |
| 同上，但要求<strong>跨轮保留</strong> | <code>input</code> 里的 <code>developer</code> 消息 |
| 传用户输入 | <code>input</code> 里的 <code>user</code> 消息 |
| 维护多轮历史 | <code>input</code> 里的 <code>assistant</code> 消息 |


#### API 选择:优先 Responses

OpenAI 的明确建议:

> If you're building any text generation app， we recommend using the <strong>Responses API</strong> over the older Chat Completions API. And if you're using a reasoning model， it's especially useful to migrate to Responses.
> 
> <strong>One important note is that reasoning models perform better and demonstrate higher intelligence when used with the Responses API.</strong>
> 
> ——推理模型在 Responses API 下表现更好、智能水平更高。

<strong>对 GPT-6 Astra 是硬要求</strong>:函数调用<strong>必须</strong>用 Responses，Chat Completions 不支持。


### 三、提示词结构:OpenAI 推荐的四段

<strong>这节讲什么</strong>:一条系统级提示词该分几块、按什么顺序写。OpenAI 给了一个默认骨架，照它搭最省事。

> In general， a developer message will contain the following sections， usually in this order (though the exact optimal content and order may vary by which model you are using):
> 
> - <strong>Identity:</strong> Describe the purpose， communication style， and high-level goals of the assistant.
> - <strong>Instructions:</strong> Provide guidance to the model on how to generate the response you want. What rules should it follow? What should the model do， and what should the model never do?
> - <strong>Examples:</strong> Provide examples of possible inputs， along with the desired output from the model.
> - <strong>Context:</strong> Give the model any additional information it might need to generate a response， like private/proprietary data outside its training data. <strong>This content is usually best positioned near the end of your prompt</strong>， as you may include different context for different generation requests.

<strong>要点</strong>:

- 顺序:<strong>Identity → Instructions → Examples → Context</strong>
- <strong>Context 放最后</strong>——因为不同请求的 context 不同，放最后有利于缓存(见第八节)
- OpenAI 强调这是"通常"的顺序，<strong>最优内容与顺序可能因模型而异</strong>

OpenAI 给的完整示例:

````text
# Identity

You are coding assistant that helps enforce the use of snake case
variables in JavaScript code， and writing code that will run in
Internet Explorer version 6.

# Instructions

* When defining variables， use snake case names (e.g. my_variable)
  instead of camel case names (e.g. myVariable).
* To support old browsers， declare variables using the older
  "var" keyword.
* Do not give responses with Markdown formatting， just return
  the code as requested.

# Examples

<user_query>
How do I declare a string variable for a first name?
</user_query>

<assistant_response>
var first_name = "Anna";
</assistant_response>
````


### 四、Markdown 与 XML 的分工

<strong>这节讲什么</strong>:两种标记法各管一件事。用错了，模型会分不清哪段是指令、哪段是要处理的材料。

> Markdown headers and lists can be helpful to <strong>mark distinct sections of a prompt， and to communicate hierarchy</strong> to the model. They can also potentially make your prompts <strong>more readable during development</strong>. XML tags can help <strong>delineate where one piece of content (like a supporting document used for reference) begins and ends</strong>. XML attributes can also be used to <strong>define metadata about content in the prompt</strong> that can be referenced by your instructions.

<strong>分工很清楚</strong>:

| <strong>工具</strong> | <strong>用途</strong> |
| --- | --- |
| <strong>Markdown</strong>(<code>&#35;</code>、列表) | 划分<strong>章节</strong>、表达<strong>层级</strong>，同时让人读起来清楚 |
| <strong>XML 标签</strong>(<code>&lt;doc&gt;</code>、<code>&lt;query&gt;</code>) | 标出一段内容的<strong>起止边界</strong> |
| <strong>XML 属性</strong>(<code>id="example-1"</code>) | 给内容<strong>附带元数据</strong>，供指令引用 |

OpenAI 的 few-shot 示例同时用了三者:

````text
# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>
````

---


### 五、Few-shot

<strong>这节讲什么</strong>:给模型看几个"输入 → 输出"的样例，比用话描述要求更管用。但推理模型上 OpenAI 建议<strong>先别给</strong>——加例子反而可能降低性能。

> Few-shot learning lets you steer a large language model toward a new task by including a <strong>handful of input/output examples</strong> in the prompt， rather than fine-tuning the model. The model implicitly "picks up" the pattern from those examples and applies it to a prompt. When providing examples， try to show a <strong>diverse range</strong> of possible inputs with the desired outputs.

<strong>推理模型上的特别提醒</strong>(OpenAI《推理模型最佳实践》):

> <strong>Try zero shot first， then few shot if needed</strong>: Reasoning models often don't need few-shot examples to produce good results， so try to write prompts without examples first. If you have more complex requirements for your desired output， it may help to include a few examples… <strong>Just ensure that the examples align very closely with your prompt instructions， as discrepancies between the two may produce poor results.</strong>
> 
> ——<strong>先试零样本，不行再加例子。</strong> 例子<strong>必须和指令高度一致</strong>，两者有出入会产生糟糕结果。
> 
> ⚠️ <strong>函数调用场景下更要注意</strong>:OpenAI 在 Function calling 指南里明确写着——<em>"Adding examples may hurt performance for reasoning models"</em>(<strong>加例子可能损害推理模型的性能</strong>)。所以给工具定义配示例时，<strong>先测再加</strong>。

---


### 六、上下文与 RAG

<strong>这节讲什么</strong>:把外部资料塞进提示词、让模型基于资料回答，这就是 RAG。这节讲它的基本概念，以及上下文窗口这个硬限制。

OpenAI 对 RAG 的定义:

> The technique of adding additional relevant context to the model generation request is sometimes called <strong>retrieval-augmented generation (RAG)</strong>. You can add additional context to the prompt in many different ways， from querying a vector database and including the text you get back into a prompt， or by using OpenAI's built-in <strong>file search tool</strong>.

<strong>上下文窗口</strong>:OpenAI 定义为模型单次生成能考虑的数据上限，以 token 计;不同模型从 10 万级到百万级不等，见各模型页。

---


### 七、把提示词当代码管理 ★

<strong>这节讲什么</strong>:提示词不能随手散落、随手改。OpenAI 要求把它当代码管——进版本库、走代码审查、配测试。<strong>而且有个硬期限:</strong><strong><code>v1/prompts</code></strong><strong> 2026-11-30 关闭。</strong>

> Store production prompts in your application code instead of creating reusable prompt objects. Code-managed prompts let you use <strong>typed inputs， code review， tests， and your normal deployment process</strong> to change model behavior.

<strong>重要时间线</strong>:

> OpenAI is deprecating reusable prompt objects in the API. Prompt creation will be de-emphasized beginning <strong>June 3， 2026</strong>， and <code>v1/prompts</code> is scheduled to shut down on <strong>November 30， 2026</strong>.

<strong>OpenAI 给出的做法</strong>:

- 把每个生产提示词放在<strong>代码管理的、有版本控制的模块</strong>里(如 <code>prompts/supportReply.ts</code>)，位置靠近它服务的功能
- 用<strong>有类型的函数参数</strong>或经过校验的输入对象，替代提示词变量
- 直接在 <code>input</code> 和 <code>instructions</code> 里把生成好的消息传给 Responses API
- 提示词改动必须<strong>配代表性 fixture、测试和评估检查</strong>
- 用 <strong>git 历史、PR 审查、发布标签、feature flag</strong> 来做审查、发布、对比和回滚

> 如果你现在还在请求里用 prompt ID 或 version，OpenAI 有专门的迁移指南 <code>guides/prompting/migrate-from-prompt-object</code>。

---


### 八、提示词缓存:结构决定成本

<strong>这节讲什么</strong>:同一段提示词反复发，缓存能省钱又省时间——但前提是你<strong>把固定不变的内容放在最前面</strong>。提示词怎么排版，直接决定账单。

> When constructing a message， you should try and keep content that you expect to <strong>use over and over</strong> in your API requests <strong>at the beginning of your prompt</strong>， and among the <strong>first API parameters</strong> you pass in the JSON request body. This enables you to maximize cost and latency savings from prompt caching.

<strong>实操含义</strong>:

- <strong>固定不变的内容放最前面</strong>(身份、指令、通用规则)
- <strong>每次变化的内容放最后</strong>(当次资料、用户提问)
- 这正好和第三节"Context 放最后"的建议互为印证

<strong>GPT-6 迁移注意事项</strong>:从 GPT-5.5 或更早版本迁移时，要把 <code>prompt&#95;cache&#95;retention</code> 换成 <code>prompt&#95;cache&#95;options.ttl</code>，值设 <code>"30m"</code>;同时要审查缓存边界和<strong>缓存写入计费</strong>的变化。

<strong>相关技巧</strong>:工具调用里若只想临时收窄可用工具，<strong>不要改动 </strong><strong><code>tools</code></strong><strong> 列表</strong>(那会破坏缓存)，改用 <code>allowed&#95;tools</code>(见第十三节)。

---


> **补充：**这里的“稳定内容放前面”主要指模型接收到的消息、文本和工具定义前缀，不是靠重排 JSON 请求对象的键就保证命中缓存。缓存命中仍有条件，读缓存也不一定零费用；原文的保留时长、参数改名及计费变化需对照实际模型文档。


### 九、推理参数

<strong>这节讲什么</strong>:控制模型"想多深、答多长"的几个旋钮。这是全文最像手册的一节，但每个参数都标了适用场景，不用背，按需查。


#### <code>reasoning.effort</code>——完整六档

<strong>合法取值</strong>(OpenAI 原文):

> Supported values are <strong>model-dependent</strong> and can include <code>none</code>， <code>minimal</code>， <code>low</code>， <code>medium</code>， <code>high</code>， <code>xhigh</code>， and <code>max</code>. Lower effort favors speed and lower token usage， while at higher effort the model thinks more completely to provide higher quality responses. The models also reason <strong>adaptively</strong> across reasoning efforts， using fewer tokens for simpler tasks and thinking harder for complex tasks.

<strong>⚠️ 例外与陷阱</strong>:

1. <strong>GPT-6 Astra 不支持 </strong><strong><code>none</code></strong>——<em>"Setting </em><em><code>reasoning.effort</code></em><em> (Responses) or </em><em><code>reasoning&#95;effort</code></em><em> (Chat Completions) to </em><em><code>none</code></em><em> returns </em><em><strong>HTTP 400</strong></em><em>."</em>
2. <strong>默认值因模型而异</strong>，不是全局统一:<code>gpt-5.5</code> 默认 <code>medium</code>;<code>gpt-5.6</code> 省略时也默认 <code>medium</code>。
3. OpenAI 明确提醒:<em>"Some models support only a subset of these values， so check the relevant model page before choosing a setting."</em>

<strong>六档完整场景表</strong>(OpenAI 原文):

| <strong>档位</strong> | <strong>OpenAI 定位与典型场景</strong> |
| --- | --- |
| <code>none</code> | <strong>延迟敏感且不需要推理或多链工具调用</strong>的任务。语音、快速信息检索、分类。(<code>gpt-5.5</code> 的延迟敏感场景，建议先试 <code>low</code>，不够再降 <code>none</code>) |
| <code>low</code> | 高效推理，延迟小幅增加。适合需要<strong>工具使用、规划、搜索、多步决策</strong>但又要控速度成本的场景:数据分析、起草、执行型编码、客服/聊天助手 |
| <code>medium</code> | <strong>质量和可靠性重要</strong>，任务涉及规划、复杂推理和判断。<strong>多数工作负载的默认配置</strong>，延迟/性能/成本帕累托曲线上的平衡点:智能体编码、研究、处理表格与幻灯片、长周期任务委派 |
| <code>high</code> | <strong>硬推理、复杂调试、深度规划</strong>，以及质量与智能比延迟更重要的高价值任务。推荐用于复杂工作流和智能体任务:智能体编码、长周期研究、知识工作。<strong>OpenAI 建议按任务复杂度同时评估 </strong><strong><code>medium</code></strong><strong> 与 </strong><strong><code>high</code></strong> |
| <code>xhigh</code> | 深度研究、异步工作流、需要<strong>长跑</strong>的智能体任务。<strong>只有在你的 evals 显示收益明显、足以 justify 额外延迟和成本时才用</strong>:安全与代码审查、企业生产力、更深入的研究任务、有挑战的编码工作流 |
| <code>max</code> | 最复杂任务的<strong>最大推理量</strong>。<strong>若你当前在用 </strong><strong><code>xhigh</code></strong><strong>，应评估 </strong><strong><code>max</code></strong><strong> 是否带来更强表现</strong> |


> **勘误：**引文列出了包含 `minimal` 的七个候选值，后表却只有六行。它们也不是所有模型都支持的统一档位表。选择前核对目标模型允许的值，不能把应用界面档位与 API 参数直接等同。


#### 把 effort 当旋钮，而不是万能药

OpenAI 在推理指南的提示建议里说得很克制:

> - Give the model the task， constraints， and desired output format.
> - <strong>Treat </strong><strong><code>reasoning.effort</code></strong><strong> as a tuning knob， not the primary way to recover quality.</strong>
> - For agentic or research-heavy workflows， define what counts as done and how the model should verify its work.
> 
> ——把 effort 当<strong>调优旋钮</strong>，<strong>不是恢复质量的主要手段</strong>;真正该做的是说清任务、约束、期望输出格式，以及定义"什么算完成"、模型如何验证自己的工作。


#### 降低首字延迟的小技巧

> For faster time to first visible token in latency-sensitive applications， <strong>ask the model to generate a short preamble</strong> before continuing with deeper reasoning.
> 
> ——在延迟敏感场景，要求模型<strong>先给一段简短前言</strong>，再继续深度推理。


#### <code>reasoning.mode</code>

> GPT-5.6 models support <code>standard</code> and <code>pro</code> reasoning modes. <strong><code>standard</code></strong><strong> is the default.</strong> Set <code>reasoning.mode</code> to <code>pro</code> for difficult tasks that need more model work and can tolerate higher latency and token usage.
> 
> Reasoning mode and reasoning effort are <strong>independent</strong>. Mode selects standard or pro execution， while <code>reasoning.effort</code> controls how much reasoning the model applies within that mode.

<strong>计费说明</strong>:pro 模式把产生最终答案所做的模型工作聚合起来，按所选模型的<strong>标准 token 费率</strong>计费——但<strong>做的工作更多，token 用量和成本都会上升</strong>。


#### <code>reasoning.context</code>

控制模型能把<strong>哪些历史推理条目</strong>渲染进下一轮采样:

| <strong>取值</strong> | <strong>行为</strong> |
| --- | --- |
| <code>auto</code> | 使用所选模型的默认值。省略该参数等同于 <code>auto</code> |
| <code>current&#95;turn</code> | 只让当前轮的推理可用，<strong>不把更早轮次的推理渲染进下一次采样</strong> |
| <code>all&#95;turns</code> | 把更早轮次中可用的、兼容的推理条目渲染进下一次采样 |

<strong>关键规则</strong>:

- <strong>GPT-5.6 家族默认 </strong><strong><code>all&#95;turns</code></strong>，更早的模型默认 <code>current&#95;turn</code>
- 响应的 <code>reasoning.context</code> 字段会报告<strong>实际生效</strong>的模式，每轮都该检查
- <code>all&#95;turns</code> 只在请求能访问到早期响应条目时才有效果(用 <code>previous&#95;response&#95;id</code>、把响应挂到会话，或手动重放完整历史)
- <strong>推理只在同一模型家族内可复用</strong>——<code>gpt-5.6-sol</code>/<code>terra</code>/<code>luna</code> 可以互相复用，但<strong>不跨 GPT-5.6 与 GPT-5.5 家族</strong>;切换家族时 API 会主动忽略不兼容的推理
- <strong>持久化的推理是"续接"而非"暴露"</strong>:推理条目始终不透明，API 不返回推理文本


#### <code>verbosity</code>

> <code>verbosity: optional "low" or "medium" or "high" or null</code>
> 
> Constrains the verbosity of the model's response. Lower values will result in more concise responses， while higher values will result in more verbose responses. <strong>The default is </strong><strong><code>medium</code></strong><strong>.</strong>

> ⚠️ 注意:当前<strong>提示词指南已不再把 verbosity 当作主要控制手段</strong>，而是推荐<strong>用提示词明确写作风格</strong>(见第十一节)。该参数在 API 参考中仍然有效。


#### <code>reasoning.summary</code>(推理摘要)

> While we don't expose the raw reasoning tokens emitted by the model， you can view a summary of the model's reasoning using the <code>summary</code> parameter.
> 
> ——模型的<strong>原始推理 token 不公开</strong>，但可以用 <code>summary</code> 参数看到<strong>推理过程的摘要</strong>。

<strong>三条要点</strong>:

- 不同模型支持不同级别:<code>concise</code>(简明)、<code>detailed</code>(详细);设成 <code>auto</code> 就能拿到该模型<strong>可用的最详细</strong>摘要
- 摘要<strong>默认不返回</strong>，必须显式开启
- 输出位置:响应里 <code>reasoning</code> output item 的 <code>summary</code> 数组

````text
response = client.responses.create(
    model="gpt-6-astra"，
    input="What is the capital of France?"，
    reasoning={
        "effort": "low"，
        "summary": "auto"，      # ← 开启推理摘要
    }，
)

print(response.output)
````

<strong>什么时候用</strong>:调提示词时想弄清"它到底怎么想的"，或者要把推理过程展示给用户看(教学、审阅类场景)。

> <strong>别搞混</strong>:<code>summary</code> 管的是<strong>推理摘要的详略</strong>;<code>text.verbosity</code> 管的是<strong>最终回答的长短</strong>。两者互不相干 —— 附录 B 里记了这条坑。


> **补充：**推理摘要不是模型内部逐字思维过程，也不能用来证明每一步都真实发生。需要审计工具执行或业务结果时，应看实际调用、返回数据与日志。


#### 中途改推理强度:<code>configuration&#95;update</code>

> Use <code>configuration&#95;update</code> to increase reasoning effort for difficult work or reduce it for routine follow-ups. Add the update between responses while <strong>leaving the request-level </strong><strong><code>reasoning.effort</code></strong><strong> unchanged</strong>. This preserves the original prompt prefix for prompt caching.

<strong>★ 但它的限制很多，不看会踩坑</strong>:

| <strong>限制</strong> | <strong>说明</strong> |
| --- | --- |
| <strong>模型范围</strong> | <strong>仅 GPT-6 Astra 支持</strong>，且<strong>仅限 standard 模式、单 agent</strong> |
| <strong>只能改什么</strong> | <strong>只改 reasoning effort</strong>，别的都不改 |
| <strong>不能相邻</strong> | <strong>不要连续放两个 </strong><strong><code>configuration&#95;update</code></strong>，API 会<strong>拒绝</strong>相邻更新 |
| <strong>不能与压缩组合</strong> | 不能与自动压缩、自动截断组合;<strong>独立的 </strong><strong><code>/responses/compact</code></strong><strong> 端点会拒绝</strong>包含这些更新的历史 |
| <strong>响应字段</strong> | 响应的 <code>reasoning.effort</code> 仍报告<strong>请求级</strong>设置，不是 update 选中的值 |
| <strong>历史保留</strong> | 用 <code>previous&#95;response&#95;id</code> 保留更新，或手动管理历史时<strong>按原位置重放</strong> |

> <strong>要和压缩一起用怎么办</strong>:显式压缩——在 <code>/responses</code> 请求里放一个 <code>compaction&#95;trigger</code> 条目;压缩完成后，<strong>在下一个用户消息前再加一条新的 </strong><strong><code>configuration&#95;update</code></strong>。


#### 上下文压缩:让长对话不撞上限

多轮对话越滚越长，迟早顶到上下文窗口上限。官方的解法是 <strong>compaction(压缩)</strong>:把历史压成一个更小的条目，同时<strong>保留继续对话所需的状态</strong>。官方已把它独立成一页《Compaction》指南。

<strong>服务端压缩(推荐)</strong>:在 <code>/responses</code> 请求里带上 <code>context&#95;management</code> 和 <code>compact&#95;threshold</code>:

````text
response = client.responses.create(
    model="gpt-6-astra"，
    input=conversation，
    store=False，
    context_management=[
        {"type": "compaction"， "compact_threshold": 200000}
    ]，
)
````

- 渲染后的 token 数<strong>跨过阈值</strong>时，服务端自动跑一次压缩
- <strong>不需要</strong>再单独调 <code>/responses/compact</code>
- 响应流里会多出一个<strong>加密的 compaction item</strong>
- 这个条目<strong>不透明、不是给人读的</strong>，只负责把关键状态和推理带到下一轮

<strong>两种续接方式，选一种走到底</strong>:

| <strong>方式</strong> | <strong>怎么做</strong> |
| --- | --- |
| <strong>无状态数组链</strong> | 把输出条目(含 compaction item)追加进下一轮的 <code>input</code> |
| <strong><code>previous&#95;response&#95;id</code></strong> | 每轮只传新的用户消息，把 ID 续下去;<strong>不要手动裁剪</strong> |

<strong>延迟技巧</strong>:走无状态数组链时，可以<strong>丢掉最近一次 compaction item 之前的条目</strong> —— 它已经带着继续对话所需的上下文，留着只会让请求变大。

> <strong>ZDR</strong>:<code>store=False</code> 时，服务端压缩对零数据保留(ZDR)是友好的。
> 
> <strong>要显式控制</strong>:用独立的 <code>/responses/compact</code> 端点，或者在 <code>/responses</code> 请求里手动插一个 <code>compaction&#95;trigger</code> 条目 —— 也就是上面「中途改推理强度」里提到的那种写法。


> **补充：**`store=False` 不等于账号已启用零数据保留（ZDR），也不代表所有服务端日志自动消失。保留策略、ZDR 资格及所用功能的兼容性需要单独确认。


#### 成本控制

| <strong>手段</strong> | <strong>说明</strong> |
| --- | --- |
| <code>max&#95;output&#95;tokens</code> | 限制模型生成的总 token 数——<strong>含推理 token、可见输出 token 和非可见格式化 token</strong> |
| <strong>预留空间</strong> | <strong>OpenAI 建议在开始试验这些模型时，为推理和输出预留至少 25，000 tokens</strong>;熟悉自己提示词的实际推理 token 用量后再调整 |
| 监控 | 实际推理 token 数在响应 <code>usage</code> 对象的 <code>output&#95;tokens&#95;details</code> 里可见 |

<strong> 一个会白花钱的坑</strong>:如果生成 token 撞上上下文窗口上限或你设的 <code>max&#95;output&#95;tokens</code>，会收到 <code>status: incomplete</code> 且 <code>incomplete&#95;details.reason</code> 为 <code>max&#95;output&#95;tokens</code>——<strong>这可能在任何可见输出产生之前就发生，意味着你付了输入和推理 token 的钱，却没拿到任何可见回复</strong>。

---


### 十、推理模型 vs GPT 模型:提示方式不同

<strong>这节讲什么</strong>:同一段提示词，给推理模型和给普通 GPT 模型，效果可能天差地别。OpenAI 用"资深同事 vs 初级同事"打了个比方，很好记。

OpenAI 给了一个特别好用的类比:

> - A reasoning model is like a <strong>senior co-worker</strong>. You can give them a goal to achieve and trust them to work out the details.
> - A GPT model is like a <strong>junior coworker</strong>. They'll perform best with explicit instructions to create a specific output.
> 
> ——推理模型像<strong>资深同事</strong>:给目标，细节他自己搞定。<br>——GPT 模型像<strong>初级同事</strong>:需要明确、具体的指令。

<strong>给推理模型写提示词</strong>(OpenAI 要点)：

- 不要写思维链

    原文：

    Since these models perform reasoning internally， prompting them to 'think step by step' or 'explain your reasoning' is unnecessary.
- 保持简单直接

    原文：

    The models excel at understanding and responding to brief， clear instructions.
- 用分隔符

原文：

Use delimiters like markdown， XML tags， and section titles to clearly indicate distinct parts of the input.

- 先零样本

    原文：

    Reasoning models often don't need few-shot examples… try to write prompts without examples first.
- 给具体约束

    原文：

    If there are ways you explicitly want to constrain the model's response， explicitly outline those constraints.
- 明确成功标准

原文：

Give very specific parameters for a successful response， and encourage the model to keep reasoning and iterating until it matches your success criteria.

<strong>另一种表述</strong>(推理指南):<em>"Reasoning-capable GPT-5 models usually work best when you give them a </em><em><strong>clear goal， strong constraints， and an explicit output contract</strong></em><em> without prescribing every intermediate step."</em>


#### <code>phase</code> 参数:长流程防早停

对使用 GPT-5.5、GPT-5.4 的长跑或工具密集流程，用 assistant 消息的 <code>phase</code> 字段:

- <code>phase: "commentary"</code>——中间更新(如工具调用前的 preambles)
- <code>phase: "final&#95;answer"</code>——完成的答案
- <strong>不要</strong>给 user 消息加 <code>phase</code>

OpenAI 说明:<code>phase</code> 在 API 层面可选，但<strong>推荐使用</strong>;<strong>缺失或丢弃 </strong><strong><code>phase</code></strong><strong> 会导致前言被当成最终答案</strong>。手动重放 assistant 历史时，要保留每个原始的 <code>phase</code> 值。

---


### 十一、GPT-6 Astra 的行为特征与对策

<strong>这节讲什么</strong>:GPT-6 Astra 有几个特有的脾气——爱问你"要不要继续"、容易被技能文件带偏、写代码时测试做过头。这节给出 OpenAI 的对策提示词，<strong>可直接抄</strong>。

OpenAI 为这个模型列了五条需要提示词来优化的行为。<strong>这一节是当前提示词工程的核心增量</strong>。


#### 主动性与执行到底

<strong>问题</strong>:模型更倾向于问你，而不是自己推断。

<strong>对策</strong>(OpenAI 原文，直接抄):

````text
You should infer the user's intent and task scope from the instructions and prior conversation context. Your job is to bias towards action and carry the user's intended task to completion.

When the user expresses intent to perform new work or fix an existing issue， persist until the user's intended goal is complete. Progress autonomously towards the user's goal … unless they are clearly destructive or irreversible.
````

````text
When the user's prompt indicates a request for action， such as "can you..."， "I want to..."， "help me..." and similar expressions， treat these as instructions to do the work and take action. Do not stop at acknowledging capability (e.g. "Yes…")， proposing a plan， or offering to continue. Do not settle for a partial or "helpful enough" solution that does not fully satisfy the user's task to save time， effort or tokens. If a task requires sustained work， complete all the necessary work until the intended outcome is fulfilled.
````

<strong>关于审批时机</strong>，OpenAI 给了个很实用的模式——<strong>先做完能做的，让用户审批一个具体的、可审查的结果</strong>:

````text
Before asking the user clarifying questions， you should complete the work that is already authorized from context and necessary to make the proposed action concrete and reviewable. The user should be approving a concrete， reviewable result. For example， before deploying a change， writing to an external application， merging a PR or publishing a site， do all the required work first so that user approval is the final step. You don't need user permission for reversible tasks， read-only actions， reviews or fixes， or anything for which authorization is provided earlier in the session or strongly implied from the task instruction.

Do not introduce unsolicited warnings， disclaimers， approval flows， or safety/compliance checklists due to hypothetical risk.
````


#### 指令遵循与"技能文件污染"

<strong>问题</strong>:模型对上下文里的信息更敏感——<strong>skill 文件或 </strong><strong><code>AGENTS.md</code></strong><strong> 里含混、冲突的指令会导致它提前停工</strong>。OpenAI <strong>强烈建议审查模型可访问的 skills 及其他文件</strong>。

解法:

````text
The user's instructions take precedence over guidelines provided in a skill. If explicit user instructions conflict with a skill's instructions， prioritize the user's instructions.
````

<strong>诊断提示词</strong>——让它说出是哪条指令让它停下的:

````text
If a skill causes you to ask for permission or confirmation， pause， leave requested work unfinished， or diverge from the user's intent， name and link to the exact SKILL.md file you read， quote the relevant instruction， and briefly explain how it applies. Distinguish explicit skill requirements from your interpretation of guidelines.
````


#### 3&#46; 人格与写作风格

<strong>问题</strong>:模型倾向于用列表、表格和 Markdown，并且<strong>可能在不同会话里重复使用同样的措辞</strong>。

<strong>若你要散文体、少格式</strong>:

````text
Default to using clear， concise paragraphs， each developing one main idea. Use lists only when the information is genuinely parallel， sequential， or easier to compare， and avoid nested lists unless the hierarchy cannot be expressed clearly in prose. Use plain， simple language: familiar words， concrete examples， and precise verbs. Prefer active voice and direct statements.

Make sure to state the main point clearly and early， then develop it with the explanation and detail the reader needs. Let each sentence build on what came before.
````

<strong>技术写作的平衡</strong>:

````text
Use plain language over jargon， and reference technical details only to the degree that it helps illustrate an idea or your work to the user. Communicate complex concepts in a clear and cohesive manner， and calibrate your writing to the level of background knowledge assumed from the user's prompt and context.
````

<strong>★ 套话黑名单</strong>(最可直接复用的一条):

````text
Avoid using slop words or phrases like "Bottom Line:" in conclusions， "delve，" "foster，" "leverage，" "it's worth noting，" "importantly，" "Question? Answer." or "This isn't about X. It's about Y."， "genuinely" or hyphenated compound descriptions and adjectives. Do not use concluding summary statements such as "In short:.."， "The simplest mental model is:...".

State the intended action directly. Avoid adding what you won't do， what will remain unchanged， or how you'll separate or categorize results. Do not use contrastive framing such as "X， not Y" or "X—not Y" that introduces an unprompted alternative that the user didn't ask about. Avoid invented compound labels like "exact-head checks" and "editorial-row layouts"， vague qualifiers， and canned transitions; use plain verbs and prepositions to state the actual relationship directly.
````

<strong>中文场景的对应词表</strong>(上面那几个是英文语境的 AI 味词;中文自有一套，而且更厚):

````text
不要使用:"值得注意的是""深入探讨""赋能""助力""总而言之""简而言之""综上所述"
不要用"首先/其次/最后"三段式套架子
不要写"让我们一起来看…""接下来我们…"这类过渡语
不要声明"本次不涉及…""以下内容保持不变""我将分三类说明"
不要用"这是X，不是Y""这不是关于X，而是关于Y"这类对比句式
不要自造复合词，不要用模糊限定词和罐头式过渡语
````


#### 子代理委派

<strong>问题</strong>:模型<strong>委派得可能比你想要得少</strong>。

````text
If at any point you can parallelize work by delegating tasks to another agent (no matter if you are the root or subagent)， you should do so using collaboration tools if it could save time or improve quality.
````

代理间消息可读性:

````text
Messages that you send to other agents and your final answer may be read by a human， so ensure they are legible. Always put proper spaces between words and/or numbers.
````


#### 测试与验证的力度校准

<strong>问题</strong>:编码任务上模型<strong>很彻底</strong>，但对小改动<strong>会跑超出需要的测试</strong>。

````text
Do not write tests for reversible， low-impact changes that mirror the implementation. If you do choose to verify your work with tests， make sure that the tests are meaningful and necessary to verify implementation.

Run tests appropriate to the change and complete required checks. Once those pass， broaden or repeat testing only when new changes， failures， or unresolved concerns justify it; otherwise， continue toward completing the task.
````

---


### 十二、Structured Outputs 实操

<strong>这节讲什么</strong>:让模型<strong>必须</strong>按你给的结构输出，不许自由发挥——喂给程序的数据走这条路。但 schema 有一堆硬性限制，<strong>不知道就直接调不通</strong>。


#### 它解决什么

> Structured Outputs is a feature that ensures the model will always generate responses that <strong>adhere to your supplied JSON Schema</strong>， so you don't need to worry about the model omitting a required key， or hallucinating an invalid enum value.

<strong>三大好处</strong>(OpenAI 原文):

1. <strong>Reliable type-safety:</strong> No need to validate or retry incorrectly formatted responses
2. <strong>Explicit refusals:</strong> Safety-based model refusals are now <strong>programmatically detectable</strong>
3. <strong>Simpler prompting:</strong> No need for strongly worded prompts to achieve consistent formatting


> **补充：**结构化输出约束的是格式，不保证字段值符合事实或业务规则。仍需处理拒绝、截断、接口错误，并校验金额、日期、权限等业务条件；“不必重试错误格式”不能理解为整个应用无需验证。


#### 两种用法，别选错

| <strong>场景</strong> | <strong>用什么</strong> |
| --- | --- |
| 把模型连到<strong>你的工具、函数、数据</strong> | <strong>Function calling</strong> |
| 需要模型<strong>回复用户时</strong>遵循某个结构 | <strong><code>text.format</code></strong><strong> 的 </strong><strong><code>json&#95;schema</code></strong> |

> <em>"If you are connecting the model to tools， functions， data， etc. in your system， then you should use function calling. If you want to structure the model's output when it responds to the user， then you should use a structured </em><em><code>text.format</code></em><em>."</em>


#### ★ 完整走一遍:从写 schema 到拿到结果

OpenAI 把这件事拆成三步，下面这份可以直接抄改。

<strong>第一步:定义 schema(标准 JSON Schema)</strong>

````text
{
  "type": "object"，
  "properties": {
    "steps": {
      "type": "array"，
      "items": {
        "type": "object"，
        "properties": {
          "explanation": { "type": "string" }，
          "output": { "type": "string" }
        }，
        "required": ["explanation"， "output"]，
        "additionalProperties": false
      }
    }，
    "final_answer": { "type": "string" }
  }，
  "required": ["steps"， "final_answer"]，
  "additionalProperties": false
}
````

对照上面「Schema 的硬性要求」那张表检查一遍:根是 <code>object</code>、每个字段都进了 <code>required</code>、每个 object 都写了 <code>additionalProperties: false</code>——三条都满足才能用。

<strong>第二步:把 schema 挂到请求上(挂在 </strong><strong><code>text.format</code></strong><strong>)</strong>

````text
response = client.responses.create(
    model="gpt-6-astra"，
    input=[
        {"role": "developer"， "content": "You are a helpful math tutor. Guide the user through the solution step by step."}，
        {"role": "user"， "content": "how can I solve 8x + 7 = -23"}，
    ]，
    text={
        "format": {
            "type": "json_schema"，
            "name": "math_response"，   # schema 的名字，必填
            "schema": {                # ← 第一步那份 schema，原样放这里
                "type": "object"，
                "properties": {
                    "steps": {
                        "type": "array"，
                        "items": {
                            "type": "object"，
                            "properties": {
                                "explanation": {"type": "string"}，
                                "output": {"type": "string"}，
                            }，
                            "required": ["explanation"， "output"]，
                            "additionalProperties": False，
                        }，
                    }，
                    "final_answer": {"type": "string"}，
                }，
                "required": ["steps"， "final_answer"]，
                "additionalProperties": False，
            }，
            "strict": True，            # 必须 True 才是真正的"强制"
        }
    }，
)

print(response.output_text)
````

> OpenAI 这个例子用的是 <code>system</code> 角色，当前推荐改用 <code>developer</code>(见第二节)。

<strong>第三步:拿到的结果长这样</strong>(OpenAI 示例返回，<code>response.output&#95;text</code> 里就是这段 JSON 文本):

````text
{
  "steps": [
    { "explanation": "Start with the equation 8x + 7 = -23."， "output": "8x + 7 = -23" }，
    { "explanation": "Subtract 7 from both sides to isolate the term with the variable."， "output": "8x = -23 - 7" }，
    { "explanation": "Simplify the right side of the equation."， "output": "8x = -30" }，
    { "explanation": "Divide both sides by 8 to solve for x."， "output": "x = -30 / 8" }，
    { "explanation": "Simplify the fraction."， "output": "x = -15 / 4" }
  ]，
  "final_answer": "x = -15 / 4"
}
````

<strong>模型返回的是文本，不是 Python 对象</strong>——要 <code>json.loads(response.output&#95;text)</code> 才能拿到上面这个 dict。


#### Structured Outputs vs JSON mode

> <strong>We recommend always using Structured Outputs instead of JSON mode when possible.</strong>

|  | <strong>Structured Outputs</strong> | <strong>JSON Mode</strong> |
| --- | --- | --- |
| 输出合法 JSON | 是 | 是 |
| <strong>遵循 schema</strong> | <strong>是</strong> | 否 |
| 启用方式 | <code>text: { format: { type: "json&#95;schema"， "strict": true， "schema": ... } }</code> | <code>text: { format: { type: "json&#95;object" } }</code> |

> ⚠️ JSON mode 有个<strong>必须知道的坑</strong>:使用 JSON mode 时，<strong>你必须在对话里(比如系统消息)明确指示模型产生 JSON</strong>。如果不加这个显式指示，模型可能生成<strong>无休止的空白流</strong>，请求会一直跑到 token 上限。为防遗漏，<strong>API 在上下文里没出现 "JSON" 字样时会直接报错</strong>。


#### ★ Schema 的硬性要求(不知道就调不通)

| <strong>要求</strong> | <strong>说明</strong> |
| --- | --- |
| <strong>根对象必须是 object</strong> | <strong>且不能是 </strong><strong><code>anyOf</code></strong>。Zod 的 discriminated union 会在顶层产生 <code>anyOf</code>，<strong>这样写不通</strong> |
| <strong>所有字段必须 </strong><strong><code>required</code></strong> | 想模拟可选字段，用<strong>联合 </strong><strong><code>null</code></strong><strong> 类型</strong> |
| <strong>必须设 </strong><strong><code>additionalProperties: false</code></strong> | Structured Outputs 只支持生成指定的键值 |
| <strong>键顺序</strong> | 输出的键顺序 = schema 里的顺序 |

<strong>规模上限</strong>:

- 最多 <strong>5000 个对象属性</strong>、<strong>10 层嵌套</strong>
- 属性名 + 定义名 + enum 值 + const 值的<strong>总字符数 ≤ 120，000</strong>
- <strong>enum 值总计 ≤ 1000 个</strong>;单个 string enum 值超过 250 个时，其<strong>总字符串长度 ≤ 15，000</strong>

<strong>支持的类型</strong>:String / Number / Boolean / Integer / Object / Array / Enum / <code>anyOf</code>

<strong>支持的约束</strong>:

- string:<code>pattern</code>、<code>format</code>(<code>date-time</code> / <code>time</code> / <code>date</code> / <code>duration</code> / <code>email</code> / <code>hostname</code> / <code>ipv4</code> / <code>ipv6</code> / <code>uuid</code>)
- number:<code>multipleOf</code>、<code>maximum</code>、<code>exclusiveMaximum</code>、<code>minimum</code>、<code>exclusiveMinimum</code>
- array:<code>minItems</code>、<code>maxItems</code>

<strong>不支持</strong>:<code>allOf</code>、<code>not</code>、<code>dependentRequired</code>、<code>dependentSchemas</code>、<code>if</code> / <code>then</code> / <code>else</code>

> <strong>微调模型额外不支持</strong>:string 的 <code>minLength</code>/<code>maxLength</code>/<code>pattern</code>/<code>format</code>、number 的 <code>minimum</code>/<code>maximum</code>/<code>multipleOf</code>、object 的 <code>patternProperties</code>、array 的 <code>minItems</code>/<code>maxItems</code>。
> 
> 若开了 <code>strict: true</code> 却传了不支持的 schema，<strong>会直接报错</strong>。

<strong>其它特性</strong>:<code>definitions</code> 支持、<strong>递归 schema 支持</strong>。


#### 处理边界情况

<strong>① 拒绝(refusal)</strong>

> When using Structured Outputs with user-generated input， OpenAI models may occasionally <strong>refuse to fulfill the request for safety reasons</strong>. Since a refusal does not necessarily follow the schema you have supplied <strong>in </strong><strong><code>response&#95;format</code></strong>， the API response will include <strong>a new field</strong> called <code>refusal</code> <strong>to indicate that the model refused to fulfill the request</strong>.

出现 <code>refusal</code> 时，要么在 UI 里展示，要么写条件逻辑处理。判断代码长这样:

````text
# 先看响应是不是被截断了
if (response.status == "incomplete"
        and response.incomplete_details.reason == "max_output_tokens"):
    raise Exception("输出被截断，响应不完整，需要调大 max_output_tokens")

# 再取出消息内容
message = next((item for item in response.output if item.type == "message")， None)
content = message.content[0] if message and message.content else None

if not content:
    raise Exception("没有返回内容")

if content.type == "refusal":
    print(content.refusal)      # 模型拒绝了，拒绝理由在这里
elif content.type == "output_text":
    print(content.text)         # 正常返回，text 里是符合 schema 的 JSON
````

<strong>② 用户输入与 schema 不兼容</strong>

> If your application is using user-generated input， <strong>make sure your prompt includes instructions on how to handle situations where the input cannot result in a valid response.</strong> The model will always try to adhere to the provided schema， <strong>which can result in hallucinations if the input is completely unrelated to the schema.</strong>

解法:在提示词里写明遇到不兼容输入时返回<strong>空参数</strong>或某句特定话。

<strong>③ 输出仍有错误</strong>

按这个顺序试:<strong>调整指令 → 在系统指令里给例子 → 拆成更简单的子任务</strong>。


#### 三条最佳实践

- <strong>schema 设计</strong>:键名清晰直观;给重要键写清楚的 title 和 description;<strong>用 evals 决定哪种结构最适合你的场景</strong>
- <strong>避免 schema 与类型定义分歧</strong>:优先用 SDK 原生 helper(Python 的 <code>pydantic.BaseModel</code>、JS 的 <code>z.object</code>);若手写 schema，加 CI 规则在 schema 或数据对象被改动时报警，或让 CI 自动互相生成
- <strong>首次请求有额外延迟</strong>:API 要处理 schema，<strong>同一 schema 的后续请求不再有额外延迟</strong>

---


### 十三、Function calling 实操

<strong>这节讲什么</strong>:让模型调用你系统里的函数(查订单、发邮件…)。重点是<strong>工具定义怎么写</strong>才让模型用得对——写得含糊，它要么乱调，要么干脆不用。


#### 术语与流程

| 概念 | 含义 |
| --- | --- |
| <strong>tool / function</strong> | 你告知模型它可用的功能 |
| <strong>tool call</strong> | 模型判断需要调用某工具时返回的特殊响应 |
| <strong>tool call output</strong> | 你的应用执行后返回给模型的结果 |

<strong>五步流程</strong>:

1. 带工具列表向模型发请求
2. 收到模型的 tool call
3. <strong>应用侧</strong>执行代码
4. 带工具输出发第二次请求
5. 收到最终响应(或更多 tool call)

<strong>五步对应的完整代码</strong>(OpenAI <code>get&#95;horoscope</code> 例子):

````text
from openai import OpenAI
import json

client = OpenAI()

# ① 定义工具列表
tools = [
    {
        "type": "function"，
        "name": "get_horoscope"，
        "description": "Get today's horoscope for an astrological sign."，
        "parameters": {
            "type": "object"，
            "properties": {
                "sign": {
                    "type": "string"，
                    "description": "An astrological sign like Taurus or Aquarius"，
                }，
            }，
            "required": ["sign"]，
        }，
    }，
]


def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."


# 会话历史，会不断变长
input_list = [{"role": "user"， "content": "What is my horoscope? I am an Aquarius."}]

# ② 第一次请求:带上工具
response = client.responses.create(
    model="gpt-6-astra"，
    tools=tools，
    input=input_list，
)

# 把模型这一轮的输出(含 function_call)追加进历史
input_list += response.output

# ③ 应用侧真正执行  +  ④ 把结果塞回历史
for item in response.output:
    if item.type == "function_call" and item.name == "get_horoscope":
        sign = json.loads(item.arguments)["sign"]   # 模型给的参数
        horoscope = get_horoscope(sign)             # 你的真实代码在这跑
        input_list.append(
            {
                "type": "function_call_output"，
                "call_id": item.call_id，            # 必须原样带回
                "output": horoscope，
            }
        )

# ⑤ 第二次请求:拿到最终回复
response = client.responses.create(
    model="gpt-6-astra"，
    instructions="Respond only with a horoscope generated by a tool."，
    tools=tools，
    input=input_list，
)

print(response.output_text)
````

<strong>两个最容易出错的点</strong>:① <strong>模型不执行任何代码</strong>——<code>item.arguments</code> 只是一段 JSON 字符串，真正干活的是你的 <code>get&#95;horoscope()</code>;② <code>call&#95;id</code> <strong>必须原样带回去</strong>，模型靠它对上这是哪一次调用的结果。


#### 函数定义的五个字段

| 字段 | 说明 |
| --- | --- |
| <code>type</code> | 固定为 <code>function</code> |
| <code>name</code> | 函数名(如 <code>get&#95;weather</code>) |
| <code>description</code> | <strong>何时、如何使用</strong>该函数 |
| <code>parameters</code> | 输入参数的 JSON Schema |
| <code>strict</code> | 是否强制 strict 模式 |

<strong>完整写出来长这样</strong>(OpenAI <code>get&#95;weather</code> 例子):

````text
{
  "type": "function"，
  "name": "get_weather"，
  "description": "Retrieves current weather for the given location."，
  "parameters": {
    "type": "object"，
    "properties": {
      "location": {
        "type": "string"，
        "description": "City and country e.g. Bogotá， Colombia"
      }，
      "units": {
        "type": "string"，
        "enum": ["celsius"， "fahrenheit"]，
        "description": "Units the temperature will be returned in."
      }
    }，
    "required": ["location"， "units"]，
    "additionalProperties": false
  }，
  "strict": true
}
````

三处细节值得盯:<code>description</code> 要写清<strong>何时用</strong>(不是只写"是什么");能穷举的值就用 <code>enum</code> 锁死(这里 <code>units</code> 只许两个值);<code>strict: true</code> &#43; 参数全部 <code>required</code> &#43; <code>additionalProperties: false</code> 三样齐了，模型才不容易乱填。


#### ★ 定义函数的最佳实践(OpenAI 原文)

1. <strong>写清楚名字、参数描述和指令</strong>

- <strong>明确描述函数的用途、每个参数的用途与格式，以及输出代表什么</strong>
- <strong>用系统提示词说明何时(以及何时不要)使用每个函数</strong>——一般来说，要<strong>精确地</strong>告诉模型该做什么
- <strong>包含示例与边界情况</strong>，尤其用来纠正反复出现的失败。<strong>注意:给推理模型加示例可能损害性能</strong>
- 对延迟加载的工具:<strong>详细指引写进函数描述，namespace 描述保持简洁</strong>(namespace 帮模型选择加载什么，函数描述帮它正确使用加载的工具)

1. <strong>套用软件工程最佳实践</strong>

- <strong>让函数可预测、符合直觉</strong>(最小惊讶原则)
- <strong>用 enum 和对象结构防止非法状态</strong>。例:<code>toggle&#95;light(on: bool， off: bool)</code> 允许非法调用
- <strong>通过"实习生测试"</strong>:一个实习生只拿到你给模型的东西，能不能正确使用这个函数?他反问你什么问题?<strong>把答案补进提示词</strong>

1. <strong>把负担从模型移到代码</strong>

- <strong>别让模型填你已经知道的参数</strong>。例:如果基于前面的菜单你已经有了 <code>order&#95;id</code>，就<strong>别设 </strong><strong><code>order&#95;id</code></strong><strong> 参数</strong>;改成定义无参的 <code>submit&#95;refund()</code>，在代码里把 <code>order&#95;id</code> 传进去
- <strong>合并总是成对调用的函数</strong>。例:如果总是在 <code>query&#95;location()</code> 之后调用 <code>mark&#95;location()</code>，就把标记逻辑并进查询函数

1. <strong>初始可用的函数数量要少</strong>

- <strong>用不同数量的函数评估你的表现</strong>
- <strong>一轮开始时可用函数尽量少于 20 个</strong>(OpenAI 说这只是"软建议")
- <strong>用 tool search</strong> 延迟加载体量大或不常用的工具，而不是一上来全暴露

1. <strong>善用 OpenAI 资源</strong>

- 在 Playground 里生成和迭代函数 schema
- 函数数量大或任务难时，考虑<strong>微调</strong>提升函数调用准确率


#### Token 用量

> Under the hood， functions are injected into the system message in a syntax the model has been trained on. This means <strong>callable function definitions count against the model's context limit and are billed as input tokens.</strong>

碰到 token 上限时:减少前置加载的函数数量、尽量缩短描述，或用 tool search 延迟加载。


#### <code>strict</code> 模式

> Setting <code>strict</code> to <code>true</code> will ensure function calls <strong>reliably adhere to the function schema</strong>， instead of being best effort. <strong>We recommend always enabling strict mode.</strong>

它基于 Structured Outputs 实现，因此继承两条要求:<strong>每个对象的 </strong><strong><code>additionalProperties</code></strong><strong> 必须为 </strong><strong><code>false</code></strong>、<strong><code>properties</code></strong><strong> 里所有字段必须标记 </strong><strong><code>required</code></strong>。

<strong>默认行为因 API 而异</strong>:

- 发送 <code>strict: true</code> 但 schema 不满足要求 → <strong>请求被拒绝</strong>，并给出缺失约束的详情
- <strong>省略 </strong><strong><code>strict</code></strong>:Responses 请求会<strong>尝试把 schema 规范化进 strict 模式</strong>，做不到才退回 best-effort(此时响应里 <code>strict: false</code>);<strong>Chat Completions 默认是非 strict</strong>
- 想在 Responses 里明确退出 strict，<strong>显式设 </strong><strong><code>strict: false</code></strong>

<strong>已知限制</strong>:部分 JSON Schema 特性不支持;对微调模型，schema 首次请求要额外处理(之后缓存)，<strong>且 schema 缓存不适用零数据保留(ZDR)</strong>。


#### <code>tool&#95;choice</code> 四种模式

| <strong>模式</strong> | <strong>行为</strong> |
| --- | --- |
| <code>"auto"</code> | <strong>(默认)</strong> 调用零个、一个或多个函数 |
| <code>"required"</code> | 调用一个或多个函数 |
| <code>{"type": "function"， "name": "get&#95;weather"}</code> | <strong>强制调用指定的某一个</strong>函数 |
| <code>allowed&#95;tools</code> | <strong>收窄</strong>模型可调用的工具子集 |
| <code>"none"</code> | 等同于不传函数 |

> <strong><code>allowed&#95;tools</code></strong><strong> 的正确用途</strong>:只想在某次请求里开放子集、又<strong>不想改动传入的 </strong><strong><code>tools</code></strong><strong> 列表</strong>时用它——这样能<strong>最大化提示词缓存的收益</strong>(改 <code>tools</code> 会破坏缓存)。


#### 并行调用

- GPT-5 起，<strong>在有内置工具可用的前提下</strong>，函数可以并行调用;<strong>内置工具不能放进并行函数调用批次</strong>
- 想禁止并行，设 <code>parallel&#95;tool&#95;calls: false</code>(保证零个或一个工具被调用)
- ⚠️ 微调模型一轮调用多个函数时，<strong>这些调用的 strict 模式会被禁用</strong>
- ⚠️ <code>gpt-4.1-nano-2025-04-14</code> 这个快照在开启并行时<strong>可能对同一工具产生多次调用</strong>，建议该快照关闭并行


> **补充：**“模型一次返回多个工具调用”与“应用实际并行执行”是两个步骤。是否允许并行取决于接口和工具组合，并不以必须先有内置工具为前提；相互依赖的操作应顺序执行。示例第二次请求也可能继续返回工具调用，完整应用需持续处理到结束或达到限制。


#### 处理调用与结果

- 模型的 <code>output</code> 数组里会有 <code>type: "function&#95;call"</code> 的条目，含 <code>call&#95;id</code>、<code>name</code>、JSON 编码的 <code>arguments</code>
- <strong>响应可能包含零个、一个或多个调用——按"有多个"来写代码</strong>
- 结果作为字符串传回(<code>function&#95;call&#95;output</code>)，格式自定(JSON、错误码、纯文本都行);<strong>无返回值的函数(如 </strong><strong><code>send&#95;email</code></strong><strong>)返回 </strong><strong><code>"success"</code></strong><strong> 之类的字符串</strong>
- 返回图片或文件时，可以传<strong>图片/文件对象数组</strong>而不是字符串
- <strong>推理模型</strong>:模型响应里随工具调用一起返回的 <strong>reasoning items 必须原样传回</strong>


#### 大规模工具集:tool search 与 namespaces

- <strong>tool search</strong>:让模型自己搜索相关工具、加入上下文再使用——<strong>仅 </strong><strong><code>gpt-5.4</code></strong><strong> 及之后支持</strong>
- <strong>namespaces</strong>:按域分组相关工具(如 <code>crm</code>、<code>billing</code>、<code>shipping</code>)，在模型需要区分服务不同系统的工具时特别有用


#### 自定义工具与 CFG(了解即可)

除 JSON Schema 驱动的函数外，还有<strong>自定义工具</strong>:模型可以给你的工具传<strong>任意字符串</strong>(比如一段 Python 代码)，不必包成 JSON。这类工具可以用 <strong>context-free grammar(CFG)</strong> 约束输出，支持 <code>lark</code> 和 <code>regex</code> 两种语法。

<strong>踩坑提示</strong>(精简自 OpenAI troubleshooting):

- <strong>终端(terminal)优先</strong>:词法分析器<strong>先于</strong>语法规则运行，<strong>贪婪匹配、最长匹配获胜</strong>——想"把一段自由文本夹在两个锚点之间"，要写成<strong>一个大的正则终端</strong>，而不是拆到多条规则里
- <strong>不要用无界的 </strong><strong><code>%ignore</code></strong>——可能让语法过于复杂，或让模型跑出分布
- <strong>正则用的是 Rust regex crate 语法</strong>，不是 Python 的 <code>re</code>
- 正则模式<strong>必须写在一行</strong>，换行用 <code>&#92;n</code> 转义;模式<strong>不要包在 </strong><strong><code>//</code></strong><strong> 里</strong>
- 模型"跑出分布"时(表现为输出异常冗长或重复，语法合法但语义错误):<strong>收紧语法 → 迭代提示词与工具描述 → 提高 reasoning effort</strong>

---


### 十四、迁移到 GPT-6 Astra 的检查清单

<strong>这节讲什么</strong>:从旧模型换到 GPT-6 Astra 要改哪些东西。OpenAI 列了 8 条，照单勾就行。<strong>正在换模型的直接看这节。</strong>

OpenAI <code>migration-quickstart</code> 原文条目:

| &#35; | 检查项 |
| --- | --- |
| 1 | <strong>模型</strong>:<code>model</code> 设为 <code>gpt-6-astra</code> |
| 2 | <strong>推理档位</strong>:当前用 <code>none</code> 或 <code>minimal</code> 的，<strong>先用 </strong><strong><code>low</code></strong><strong> 起步再对比</strong>;其余<strong>保持现有有效档位</strong>。Responses 用 <code>reasoning.effort</code>，Chat Completions 用 <code>reasoning&#95;effort</code> |
| 3 | <strong>工具调用</strong>:用 <strong>Responses API</strong>。GPT-6 Astra 支持 Chat Completions，但<strong>工具调用必须用 Responses</strong> |
| 4 | <strong>移除不支持的参数</strong>:<code>temperature</code>、<code>top&#95;p</code>、<code>top&#95;logprobs</code>;Chat Completions 还要移除 <code>logprobs</code>;Responses 从 <code>include</code> 里移除 <code>message.output&#95;text.logprobs</code> |
| 5 | <strong>Fast mode</strong>:EU 数据驻留场景用 Standard 处理;<code>service&#95;tier: "fast"</code> / <code>"priority"</code> 在 EU 数据驻留下不可用 |
| 6 | <strong>中途改档位</strong>:用 <code>configuration&#95;update</code> 条目，保持请求级 <code>reasoning.effort</code> 不变以保留缓存前缀 |
| 7 | <strong>提示词缓存</strong>:从 GPT-5.5 或更早迁移时，<code>prompt&#95;cache&#95;retention</code> → <code>prompt&#95;cache&#95;options.ttl = "30m"</code> |
| 8 | <strong>多余的审批停顿</strong>:若模型反复请求批准，用第十一节的"主动性与执行到底"提示词 |

<strong>便捷路径</strong>:Codex 可按这份指南自动应用改动，用 <code>openai-docs</code> skill:

````text
$openai-docs migrate this project to GPT-6 Astra
````


### 十五、评估与迭代

<strong>这节讲什么</strong>:怎么知道提示词是改好了还是改坏了——<strong>靠测试，不靠感觉</strong>。这节给最小可行做法。

OpenAI 给出的理由:

> Because the content generated from a model is <strong>non-deterministic</strong>… Even <strong>different snapshots of models within the same family could produce different results</strong>.

<strong>三个动作</strong>:

1. <strong>生产环境钉住具体快照号</strong>(如 <code>gpt-5.5-2026-04-23</code>)，不要用浮动别名
2. <strong>建评估套件</strong>衡量提示词行为——迭代时、换模型版本时都要跑
3. 提示词改动<strong>走代码审查流程</strong>，配代表性 fixture 与评估检查


#### 想做得更系统:官方《Evals》的四步

上面是"最小做法"。要做成自动化、可复跑的评估，官方给了一条完整路径(以"把 IT 工单分类"为例):

| <strong>步骤</strong> | <strong>做什么</strong> |
| --- | --- |
| <strong>1&#46; 定义任务</strong> | 写清任务与提示词 —— 比如"把工单分类成 Hardware / Software / Other，只回一个词" |
| <strong>2&#46; 配数据源</strong> | 用 <code>data&#95;source&#95;config</code> 指定测试数据从哪来 |
| <strong>3&#46; 定评判标准</strong> | 用 <code>testing&#95;criteria</code> 说明"什么样的输出算对" |
| <strong>4&#46; 跑并分析</strong> | 上传测试数据 → 创建 eval run → 看结果 |

<strong>第 3 步是关键</strong>。官方例子里用的是 <code>string&#95;check</code> 打分器 —— 把模型输出和你标注的正确答案做<strong>精确比对</strong>:

````text
{
  "type": "string_check"，
  "name": "Category string match"，
  "input": "{{ sample.output_text }}"，
  "operation": "eq"，
  "reference": "{{ item.category }}"
}
````

<code>{{ sample.output&#95;text }}</code> 是模型这次的输出，<code>{{ item.category }}</code> 是你数据里标注的正确答案 —— 两边相等才算过。这套写法适合<strong>有唯一正确答案</strong>的任务:分类、信息抽取、判分。

---


### 附录 A:能力速查

| <strong>我要…</strong> | <strong>用什么</strong> |
| --- | --- |
| 让模型更主动、别老问 | 第十一节第 1 条的提示词块 |
| 让回复更短 | <code>verbosity: "low"</code> <strong>或</strong>用提示词约束写作风格 |
| 调思考深度 | <code>reasoning.effort</code>(六档，注意 GPT-6 <strong>不支持 </strong><strong><code>none</code></strong>) |
| 中途改思考深度又不破坏缓存 | <code>configuration&#95;update</code>(仅 GPT-6 Astra、standard、单 agent，且不能相邻) |
| 看模型"怎么想的" | <code>reasoning={"summary": "auto"}</code>(推理摘要，和 <code>text.verbosity</code> 不是一回事) |
| 长对话撞上下文上限 | 服务端压缩:<code>context&#95;management</code> &#43; <code>compact&#95;threshold</code> |
| 控制推理成本 | <code>max&#95;output&#95;tokens</code> &#43; <strong>为推理预留 ≥25，000 tokens</strong> |
| 固定输出格式给程序用 | Structured Outputs(<code>text.format</code> 的 <code>json&#95;schema</code>) |
| 把模型连到我的系统 | Function calling + <strong><code>strict: true</code></strong> |
| 临时收窄可用工具又不破坏缓存 | <code>allowed&#95;tools</code> |
| 工具太多装不下 | <code>tool search</code>(gpt-5.4+)、namespaces |
| 给资料并防编造 | <code>&lt;资料&gt;</code> 标签 + "只用资料" + "没有就说不知道" |
| 省成本/降延迟 | 稳定内容前置(缓存)、<code>low</code> 档、控制上下文、延迟加载工具 |
| 长流程防早停 | <code>phase: "commentary"</code> / <code>"final&#95;answer"</code>(GPT-5.5/5.4) |
| 排查"模型不听话" | 检查 skill / <code>AGENTS.md</code> 里是否有冲突指令 |


### 附录 B:易踩的坑

| <strong>坑</strong> | <strong>说明</strong> |
| --- | --- |
| 还在用 <code>system</code> 角色 | 推理模型自 <code>o1-2024-12-17</code> 起用 <strong><code>developer</code></strong> |
| 传了 <code>temperature</code>/<code>top&#95;p</code> | GPT-6 Astra <strong>要移除</strong> |
| 给 GPT-6 传 <code>none</code> | 返回 <strong>HTTP 400</strong> |
| 以为 <code>instructions</code> 会跨轮保留 | <strong>不会</strong>——它只对当前请求生效 |
| 把可变内容放在提示词开头 | <strong>破坏缓存</strong>;可变内容应放最后 |
| 假设默认 effort 统一 | <strong>因模型而异</strong>;<code>gpt-5.5</code>/<code>gpt-5.6</code> 默认 <code>medium</code> |
| 给推理模型加 few-shot | 先<strong>零样本</strong>，不够再加;且例子必须与指令一致 |
| 以为 <code>max&#95;output&#95;tokens</code> 只限可见输出 | <strong>推理 token 和格式化 token 也算在内</strong> |
| 没给推理预留空间 | <strong>可能付了钱却拿不到任何可见回复</strong>(<code>status: incomplete</code>) |
| 用 Zod 的 discriminated union 做根 schema | 根对象<strong>不能是 </strong><strong><code>anyOf</code></strong>，会直接报错 |
| schema 里写了 <code>allOf</code>/<code>if-then-else</code> | <strong>不支持</strong>，开 <code>strict</code> 会报错 |
| 用 JSON mode 却没写 "JSON" 字样 | <strong>API 直接报错</strong>;写了但不够明确还可能生成无限空白 |
| 以为 JSON mode 会遵循 schema | <strong>不会</strong>，它只保证是合法 JSON |
| 函数一上来就暴露几十个 | <strong>初始控制在 20 个以内</strong>，其余用 tool search 延迟加载 |
| 让模型填你已知的参数 | 把负担移到代码里，函数参数越少越准 |
| 改了 <code>tools</code> 列表来收窄工具 | <strong>破坏缓存</strong>，应用 <code>allowed&#95;tools</code> |
| 连续放两个 <code>configuration&#95;update</code> | <strong>API 拒绝相邻更新</strong> |
| <code>configuration&#95;update</code> 配自动压缩 | <strong>不兼容</strong>;<code>/responses/compact</code> 会拒绝这类历史 |
| 推理模型下没把 reasoning items 传回 | 工具调用时<strong>必须原样传回</strong>，否则推理要重来 |
| 用 Chat Completions 做 tool calling | GPT-6 Astra <strong>必须用 Responses</strong> |
| 还在用 prompt ID / 版本对象 | <code>v1/prompts</code> <strong>2026-11-30 关闭</strong>，迁到代码里 |
